# 5: Regression and Classification

**Author:** Dr Rob Lyon 

**Version:** 1.0

## Code & License
Copyright © 2026 Robert Lyon. All Rights Reserved.

This notebook and all associated materials are the intellectual property of the author.

Permission is granted solely to read, study, and analyse this material for personal educational purposes. No other rights are granted.

Without the prior written consent of the author, you may not:

* Copy, reproduce, redistribute, publish, transmit, or display this work in whole or in part.
* Modify, adapt, transform, translate, or create derivative works based on this material.
* Incorporate any portion of this work into another project, publication, product, model, dataset, or codebase.
* Use this material for commercial purposes.
* Remove or alter this copyright notice.

All intellectual property rights, including copyright and any derivative rights, remain exclusively vested in the author.

Access to this material does not constitute a transfer of ownership, license, or any other intellectual property rights except as expressly stated above.

---

**Note:**

This notebook is designed to be viewed from within JupyterLab. The Table of Contents and "Back to Table of Contents" links rely on JupyterLab's anchor handling to jump between sections.

If you open this notebook in another tool, such as PyCharm or Visual Studio Code, these anchor links may not work as expected. You can still navigate the notebook in those tools by using the headings directly, most editors provide an outline or contents panel that lists them for you.

---
## Table of Contents

1. [Introduction](#1.-Introduction)
2. [Useful Links](#2.-Useful-Links)
3. [Libraries and Environment Setup](#3.-Libraries-and-Environment-Setup)
4. [Simple Linear Regression](#4.-Simple-Linear-Regression)
    - [4.1 The Model](#4.1-The-Model)
    - [4.2 Generating a Synthetic Dataset](#4.2-Generating-a-Synthetic-Dataset)
    - [4.3 Fitting - Finding the Best Line](#4.3-Fitting---Finding-the-Best-Line)
    - [4.4 The Least-Squares Method - Finding the Optimal Parameters](#4.4-The-Least-Squares-Method---Finding-the-Optimal-Parameters)
    - [4.5 Evaluating the Fit](#4.5-Evaluating-the-Fit)
    - [4.6 Linear Regression Example](#4.6-Linear-Regression-Example)
    - [4.7 The Four Key Assumptions of Linear Regression](#4.7-The-Four-Key-Assumptions-of-Linear-Regression)
5. [Multiple Linear Regression](#5.-Multiple-Linear-Regression)
    - [5.1 The Model](#5.1-The-Model)
    - [5.2 Fitting and Interpreting a Multiple Regression Model](#5.2-Fitting-and-Interpreting-a-Multiple-Regression-Model)
6. [From Regression to Classification](#6.-From-Regression-to-Classification)
    - [6.1 Why Linear Regression Cannot Reliably Classify](#6.1-Why-Linear-Regression-Cannot-Reliably-Classify)
    - [6.2 The Sigmoid Function](#6.2-The-Sigmoid-Function)
    - [6.3 The Logit Transformation](#6.3-The-Logit-Transformation)
    - [6.4 Logistic Regression with scikit-learn](#6.4-Logistic-Regression-with-scikit-learn)
    - [6.5 Why Least Squares Fails for Logistic Regression](#6.5-Why-Least-Squares-Fails-for-Logistic-Regression)
    - [6.6 Maximum Likelihood Estimation](#6.6-Maximum-Likelihood-Estimation)
7. [Summary](#7.-Summary)
8. [References](#8.-References)
---

## 1. Introduction

This notebook seeks to connect the ideas underpinning linear algebra and optimisation to classification.

The main aim is to develop an understanding of regression, both as a predictive tool in its own right and as a link to the topic of classification. We begin in this notebook with simple linear regression and then move on to multiple linear regression. Next we build an understanding of the least-squares solution and discuss model evaluation. We then examine the fundamental limitations of linear regression for classification tasks, introduce the sigmoid function and the logit transformation, and build up an understanding of logistic regression step by step. This culminates in Maximum Likelihood Estimation (MLE) as the optimisation objective.

### Learning Objectives

By the end of this notebook you should be able to:

- State the equation of a simple linear regression model and interpret the parameters $\beta_0$ and $\beta_1$.
- Derive the closed-form least-squares solution for $\beta_0$ and $\beta_1$ and explain why squaring the residuals is preferable to summing them directly.
- Compute and interpret the four key regression evaluation metrics: MSE, RMSE, MAE, and R².
- State and check the four assumptions of ordinary least squares regression.
- Extend simple regression to multiple predictors and interpret the marginal contribution of each coefficient.
- Explain why applying linear regression directly to a binary label is problematic.
- Define the sigmoid function, describe its domain and range, and connect it to the logit transformation.
- Describe the full logistic regression pipeline: linear combination → log-odds → sigmoid → predicted probability → classification.
- Explain what Maximum Likelihood Estimation is and how it gives rise to the binary cross-entropy loss.


---

## 2. Useful Links

| Link | Description |
|------|-------------|
| [scikit-learn: linear models](https://scikit-learn.org/stable/modules/linear_model.html) | Official documentation for `LinearRegression` and `LogisticRegression`, including their mathematical formulations, parameters, and usage examples. |
| [scikit-learn: model evaluation](https://scikit-learn.org/stable/modules/model_evaluation.html) | Covers all metrics available in `sklearn.metrics`, including MSE, RMSE, MAE, R², and classification metrics such as accuracy and the confusion matrix. |
| [NumPy statistics reference](https://numpy.org/doc/stable/reference/routines.statistics.html) | Reference for NumPy's statistical functions, including `np.mean`, `np.var`, `np.cov`, and related routines used throughout this notebook. |
| [Matplotlib tutorials](https://matplotlib.org/stable/tutorials/index.html) | Step-by-step guides for creating and customising plots in Matplotlib, useful for understanding and adapting the visualisation code in this notebook. |
| [Wikipedia: Logistic regression](https://en.wikipedia.org/wiki/Logistic_regression) | A clear high-level overview of logistic regression, covering its history, the logit transformation, and the relationship to the sigmoid function. |
| [Wikipedia: Maximum likelihood estimation](https://en.wikipedia.org/wiki/Maximum_likelihood_estimation) | An accessible explanation of MLE, including the log-likelihood trick and the relationship to minimising the negative log-likelihood. |



---

## 3. Libraries and Environment Setup

🔙 [Back to Table of Contents](#Table-of-Contents)

### 3.1 Working Environment

To run the code in this notebook, you will need a Python environment with the required libraries listed below installed.

The easiest way to get started is to use the **Project IO** Docker container, which provides a fully configured and tested environment containing all required Python packages and supporting dependencies. Instructions for downloading and running the container can be found in the **project-io** GitHub repository:

https://github.com/scienceguyrob/project-io

Using the Docker container ensures that the notebooks run in exactly the same environment in which they were developed and tested.

If you prefer not to use Docker, a good alternative is to install the [Anaconda Distribution](https://www.anaconda.com/products/distribution). You can then create a new Python environment and install the required libraries using either `conda install` or `pip install`.

---

### 3.2 Libraries Used in This Notebook

All sections of this notebook rely on at least one external library. The table below lists each library, its purpose, and a link to its documentation.

| Library | Purpose | Documentation |
|---------|---------|---------------|
| **NumPy** (`numpy`) | Provides fast numerical arrays and mathematical operations. Used throughout for array arithmetic, random number generation, and computing statistics such as means and variances. | [numpy.org](https://numpy.org/doc/stable/) |
| **Matplotlib** (`matplotlib.pyplot`) | The standard Python plotting library. Used to create scatter plots, line plots, histograms, bar charts, and contour plots throughout this notebook. | [matplotlib.org](https://matplotlib.org/stable/) |
| **pandas** (`pandas`) | Provides the `DataFrame` structure used in Section 5 to organise the multi-feature house price dataset and inspect it with `.describe()`. | [pandas.pydata.org](https://pandas.pydata.org/docs/) |
| **scikit-learn** (`sklearn`) | Provides `LinearRegression`, `LogisticRegression`, `train_test_split`, and associated metrics. Used from Section 4.3 onwards. | [scikit-learn.org](https://scikit-learn.org/stable/) |
| **ipywidgets** (`ipywidgets`) | Adds interactive UI controls (sliders, dropdowns, etc.) to Jupyter notebooks. We use it to create sliders for adjusting plot parameters in real time. | [ipywidgets.readthedocs.io](https://ipywidgets.readthedocs.io/en/stable/) |
| **ipympl** (`matplotlib widget` backend) | Enables interactive, live-updating Matplotlib figures inside Jupyter. Required for smooth slider updates without redrawing the entire plot. | [matplotlib.org/ipympl](https://matplotlib.org/ipympl/) |

### 3.3 Data

This notebook uses only synthetic datasets generated in code, so no external data files are required. Each dataset is described when it is introduced.

### 3.4 Imports

All library imports for this notebook are placed in the cell below. This is a deliberate best practice: keeping all imports in one place at the top of a notebook makes it easy to see at a glance what is required and avoids confusing errors that arise when an import cell is skipped.

> ⚠️ **You must run the cell below before executing any other code in the notebook.** If you skip this cell, subsequent cells will raise a `NameError`.


In [ ]:
# Listing 1.
# ─────────────────────────────────────────────────────────────────────────────
# All library imports for this notebook are placed here for clarity.
# Run this cell first before executing any code.
# ─────────────────────────────────────────────────────────────────────────────
import numpy as np                          # Numerical arrays and mathematical functions
import matplotlib.pyplot as plt             # Plotting and visualisation

import matplotlib
matplotlib.rcParams['figure.max_open_warning'] = 50

import pandas as pd                         # DataFrame for organising tabular data

from sklearn.model_selection import train_test_split   # Splitting data into train/test sets
from sklearn.linear_model import LinearRegression      # Ordinary least-squares linear regression
from sklearn.linear_model import LogisticRegression    # Logistic regression classifier
from sklearn.metrics import (                          # Standard evaluation metrics
    mean_squared_error,
    mean_absolute_error,
    r2_score,
    accuracy_score,
    confusion_matrix,
)

# Seeded random number generator for reproducible synthetic data
rng = np.random.default_rng(42)

# Confirm successful import
print('Libraries loaded successfully.')


---


## 4. Simple Linear Regression

🔙 [Back to Table of Contents](#Table-of-Contents)

In earlier notebooks you worked with straight lines as decision boundaries, using the equation $y = mx + c$ to separate two classes. In Notebook 3 you explored features, distributions, and how variables relate to each other. In Notebook 4 you built a complete optimisation pipeline: defining a loss function, searching for the parameters that minimise it, and understanding how gradient descent navigates that search.

Here I bring all of these threads together. The central question shifts from **classification**, predicting which discrete *category* a data point belongs to, to **regression**, predicting a *continuous numerical value*. Rather than asking "is this fruit an apple or an orange?", we now ask "what price will this house sell for?" or "how much will a patient's blood pressure change with age?". We do come back to classification later. For now the distinction between prediction and classification matters because for prediction:

- The **output** is no longer a label from a fixed set: it is a number that can take any value along a scale
- The **loss function** changes: instead of counting misclassifications, we measure how far our numerical predictions are from the true values
- The **model** changes: instead of a threshold or a boundary, we fit a line (or curve) through the data

Despite these differences, the underlying framework is identical to what we did in Notebook 4: define a model with tunable parameters, choose a loss function, and find the parameter values that minimise it. 

### 4.1 The Model

Imagine you are an estate agent trying to estimate the sale price of a house. You have records of 60 past sales, and for each one you know the floor area in m² and the price it sold for in £k. You notice that larger houses tend to sell for more, and you want to capture that relationship mathematically so that when a new house comes on the market, you can feed in its floor area and get a price estimate out.

This is the core idea of regression. You use the data you already have, where you know both the input ($x$ = floor area) and the true output ($y$ = sale price), to **fit a line** that best describes the relationship between the two. Once the line is fitted, you can use it to **predict** the output for new inputs where you do not know the answer yet. This line is a model that represents our understanding of the data: our model of the relationship it contains between the two variables floor area and sale price.

<br>

The simplest regression model represents this relationship as a straight line:

$$y = \beta_0 + \beta_1 x$$

This is the same equation you encountered during earlier notebooks, $y = c + mx$, written with Greek letters (beta) used commonly throughout statistics and machine learning. Here:

- $\beta_0$ is the **intercept**: the predicted price when floor area is zero (the point where the line crosses the y-axis)
- $\beta_1$ is the **slope**: how much the predicted price increases for every additional m² of floor area

These two values are the model's **parameters**. Before we look at any data, $\beta_0$ and $\beta_1$ are unknown. **Training** is the process of finding the values of $\beta_0$ and $\beta_1$ that make a line fit the data as well as possible. In other words, it's about finding the line that comes closest to representing the relationship between actual sale prices and floor area across all houses in our records.

Once $\beta_0$ and $\beta_1$ have been determined from the training data, the line is fixed. We can then feed in the floor area of any new house, one that was not in our original records, and the model will produce a specific price estimate. To do this, we simply trace from the floor area value on the x-axis vertically upwards until it meets the line. Once it meets the line, we trace a new line left from that point until we hit the y-axis. The point we hit on the y-axis is our prediction. We write this predicted value as $\hat{y}$ (read "y-hat") to distinguish it from the true value $y$.

$$\hat{y}_i = \beta_0 + \beta_1 x_i$$

For example, if fitting the line to our 60 sales gives us $\beta_0 = 50$ and $\beta_1 = 3$, then a new house with floor area $x = 80\text{ m}^2$ would be predicted to sell for:

$$\hat{y} = 50 + 3 \times 80 = 290 \text{ £k}$$

Run the code below to see this visualised.

In [ ]:
# Listing 2.
%matplotlib widget
from visualisations.Figure_0 import show
show()

The model has taken an input it has never seen before and produced a specific numerical estimate. This is what we mean by prediction.

#### Residuals: measuring how wrong each prediction is

For the houses in our training data, we know both the true price $y_i$ and what any line we might fit predicts $\hat{y}_i$. The gap between them is called the **residual**. Here $r_i$ is the residual for the true price of example $i$ and what a line predicted for example $i$:

$$r_i = y_i - \hat{y}_i$$

A positive residual means the house sold for *more* than the line predicted, so the model underestimated. A negative residual means it sold for *less*, so the model overestimated. No line will fit every data point perfectly because real data always contains noise and variation that a simple straight line cannot capture. Training is therefore the process of finding the $\beta_0$ and $\beta_1$ that make the residuals as small as possible *on average* across all training examples. The focus is not on finding parameters that yield a residual of zero for any one point, but on making the total summed residual as small as possible across all of them.

The figure below shows a synthetic house-price dataset with the fitted regression line already drawn. The line is annotated with the values of $\beta_0$ and $\beta_1$ found by fitting: these are the parameters the model has learned from the 60 training examples.

To see how the model makes a prediction for a house it has never seen before:

1. Type a floor area (in m²) into the input box, any value between 40 and 160
2. Click **Predict price**

The plot will draw two green dashed lines: one rising vertically from your chosen floor area up to the fitted line, and one running horizontally across to the price axis. The point where the vertical line meets the fitted line is the model's prediction; read off the y-axis to see the estimated sale price in £k.

Click **Clear** to remove the prediction lines and try a different value. Notice how the predicted price changes as you increase or decrease the floor area: each additional m² adds exactly $\beta_1$ £k to the predicted price, regardless of where you start. This is the slope in action.

In [ ]:
# Listing 3.
%matplotlib widget
from visualisations.Figure_1 import show
show()

We'll look at how the fitting works practically soon.

### 4.2 Generating a Synthetic Dataset

Before we can fit a regression model, we need data. In a real project you would load a dataset from a file or database. Here we **generate synthetic data**: data we create artificially using a known mathematical relationship plus some random noise. This has two advantages for learning: we know exactly what the true relationship is (so we can check whether our model recovers it), and we can generate as many observations as we like.

The approach is straightforward and one you can reuse whenever you need a controlled dataset to experiment with:

1. **Define the true relationship**: decide what the underlying model is. Here we use $\text{price} = 1.8 \times \text{area} + 50$, meaning every additional m² adds £1.8k to the price, and the base price is £50k. In a real project you wouldn't normally know this relationship in advance, of course.
2. **Generate the input feature**: draw values of $x$ (floor area) randomly from a uniform distribution between 40 and 160 m² using `rng.uniform()`.
3. **Compute the output**: apply the true relationship to each $x$ value to get the "ideal" price.
4. **Add noise**: real data is never perfectly linear, so we add Gaussian noise using `rng.normal(0, 18, n)`, which adds a random perturbation drawn from a normal distribution with mean 0 and standard deviation 18. This simulates the natural variation in house prices that cannot be explained by floor area alone, such as local amenities, condition of the property, market timing, and so on.

The result is a dataset that looks realistic but where we know the ground truth: $\beta_0 = 50$ and $\beta_1 = 1.8$. When we fit the model, we can compare what it learns against these true values.

#### Dataset description

| Variable | Symbol | Units | Description |
|---|---|---|---|
| Floor area | $x$ | m² | The input feature: size of the property, drawn uniformly from [40, 160] |
| Sale price | $y$ | £k | The target variable: computed from the true relationship plus Gaussian noise (σ = 18) |
| Number of observations | $n$ | — | 60 synthetic house sales |

The true (data-generating) model is:

$$y = 1.8 \times x + 50 + \varepsilon \qquad \text{where} \quad \varepsilon \sim \mathcal{N}(0, 18^2)$$

The $\varepsilon$ (epsilon) term represents noise: a random value drawn independently for each observation from a normal distribution with mean zero and standard deviation 18. The notation $\varepsilon \sim \mathcal{N}(0, 18^2)$ is read as "epsilon is drawn from a normal distribution with mean 0 and variance $18^2$". In plain English, the noise added to each house price is a random number centred on zero (so it is equally likely to push the price up or down) with a standard deviation of 18, meaning most noise values fall within roughly ±18 £k of zero.

#### Visualising the raw data

Before fitting any model, it is good practice to plot the raw scatter and visually inspect the data. The code below does this using `ax.scatter()`, which plots each observation as a dot with floor area on the x-axis and sale price on the y-axis.

What to look for in the scatter plot:

- **Is there a visible trend?** If larger floor areas tend to produce higher prices, a linear model may be appropriate.
- **How much scatter is there?** Points spread tightly around an imaginary line suggest the relationship is strong and the model will fit well; wide scatter suggests a lot of unexplained variation.
- **Are there any obvious outliers?** Unusual points far from the main cloud may warrant investigation before fitting.

In this case you should see a clear upward trend with moderate scatter, exactly what we built in, with a slope of 1.8 and noise standard deviation of 18.

**Read the code below**, try to understand how it's generating the data. 

In [ ]:
# Listing 4.
# ── Step 1: Define how many observations to generate ─────────────────────────
n = 60   # 60 synthetic house sales — enough for a clear trend without being overwhelming

# ── Step 2: Generate the input feature (floor area) ──────────────────────────
# rng.uniform(low, high, size) draws 'size' values uniformly at random between
# low and high — meaning every value in that range is equally likely.
# Here we simulate 60 houses with floor areas between 40 m² and 160 m².
area = rng.uniform(40, 160, n)

# ── Step 3: Generate the target variable (sale price) ─────────────────────────
# We apply the true underlying relationship: price = 1.8 × area + 50
# This means each extra m² adds £1.8k, and the base price is £50k.
#
# rng.normal(mean, std, size) draws 'size' values from a normal distribution
# centred on 'mean' with spread 'std'. Here mean=0 means the noise is unbiased
# (errors are equally likely to be positive or negative), and std=18 means
# most prices deviate from the true line by roughly ±18 £k.
# Adding this noise makes the data look realistic — real house prices are never
# perfectly predicted by floor area alone.
price = 1.8 * area + 50 + rng.normal(0, 18, n)

# ── Step 4: Plot the raw scatter ──────────────────────────────────────────────
# Always visualise your data before fitting any model — it helps you spot
# outliers, check whether the trend looks linear, and understand the spread.
fig, ax = plt.subplots(figsize=(8, 5))
fig.canvas.header_visible = False
fig.canvas.resizable = True
fig.canvas.toolbar_visible = True
fig.canvas.toolbar_position = 'right'

ax.scatter(
    area, price,
    color='steelblue',   # fill colour of each dot
    s=55,                # marker size (in points²)
    edgecolors='k',      # thin black border makes dots easier to distinguish
    linewidth=0.4,       # width of that border
    alpha=0.8,           # slight transparency so overlapping dots are visible
    zorder=3             # draw the dots on top of the grid lines
)

ax.set_xlabel('Floor area (m²)')
ax.set_ylabel('Sale price (£k)')
ax.set_title('Figure 2: Synthetic house price data (n = 60)')
ax.grid(True, alpha=0.25)   # light grid makes it easier to read off values
plt.tight_layout()
plt.show()

# Print a quick summary so we can confirm the data was generated as expected
print(f'Dataset: {n} observations  |  area range: {area.min():.0f}–{area.max():.0f} m²')

---

### 4.3 Fitting - Finding the Best Line

We now have a dataset and a model (our model is $y = \beta_0 + \beta_1 x$) . The question is: how do we find the values of $\beta_0$ and $\beta_1$ that make a line fit the data as well as possible?

To understand why this question is non-trivial, it helps to first see what happens when we get it wrong. We will deliberately choose *incorrect* parameter values, a rough guess, and measure the consequences. This builds intuition for what "fitting" actually means before we derive the optimal solution.

#### Starting with a guess

Suppose we guess $\beta_0 = 60$ and $\beta_1 = 1.5$. We do not need to justify these values; the point is that they are plausible but not optimal. With these guessed parameters, the model produces a predicted sale price for every house in the dataset by plugging each house's floor area into the equation. So for a house with a floor area of 80 m², the model predicts:

$$\hat{y} = 60 + 1.5 \times 80 = 180 \text{ £k}$$

And for a house with a floor area of 120 m²:

$$\hat{y} = 60 + 1.5 \times 120 = 240 \text{ £k}$$

The model does this for all houses in the dataset, each one getting its own prediction based solely on its floor area. Every other factor that might influence price (location, condition, age of the property) is ignored, because our model only has one input feature.

For most houses, the predicted price will not exactly match the true price. The gap between the two, the **residual** $r_i = y_i - \hat{y}_i$, is the model's error on that individual observation. A positive residual means the house sold for more than we predicted; a negative residual means it sold for less.

If our guessed line were perfect, all residuals would be exactly zero. In practice, no single straight line can pass through every data point, because real data always contains noise and variation. The best we can do is find the line that makes the residuals collectively as small as possible.

#### Visualising the errors

The code below draws the guessed line over the scatter data and shows each residual as a vertical red line connecting the actual data point to the line. A short residual line means the prediction was close; a long one means it was far off. Scanning the plot, you should be able to see whether the line is systematically too high, too low, or simply has a slightly wrong slope.

The right panel shows the *distribution* of all residuals as a histogram. For a well-fitted line, this distribution should be:

- **Centred on zero**, meaning the model is not systematically over- or under-predicting
- **Roughly symmetric**, meaning errors above and below the line are equally likely

For our guessed line the histogram will be shifted away from zero, a clear visual signal that the parameters are not optimal and the line needs to be adjusted. The **sum of squared residuals** printed below the plot is a single number that summarises the total error. We will use exactly this quantity as our loss function in the next section to find the best possible $\beta_0$ and $\beta_1$.

Read the code below, try to understand it and execute it when ready.

In [ ]:
# Listing 5.
# ── Step 1: Set the guessed parameter values ──────────────────────────────────
# These are deliberately not the true values (β₀=50, β₁=1.8) — we are
# intentionally starting with a poor guess to illustrate what a bad fit looks like.
b0_guess = 60.0   # our guess for β₀ (intercept)
b1_guess = 1.5    # our guess for β₁ (slope — £k per m²)

# ── Step 2: Compute predicted prices under the guessed line ───────────────────
# For each house in the dataset, plug its floor area into the guessed equation:
#   ŷ_i = b0_guess + b1_guess × area_i
# This gives one predicted price per house — a numpy array of 60 values.
y_hat = b0_guess + b1_guess * area

# ── Step 3: Compute residuals ─────────────────────────────────────────────────
# The residual for each house is the difference between the true price and
# the predicted price: r_i = y_i - ŷ_i
# A positive residual means we under-predicted (the house sold for more than
# our line suggested); a negative residual means we over-predicted.
residuals = price - y_hat

# ── Step 4: Plot ──────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
fig.canvas.header_visible = False
fig.canvas.resizable = True
fig.canvas.toolbar_visible = True
fig.canvas.toolbar_position = 'right'

# ── Left panel: scatter with residuals shown as vertical lines ────────────────
ax = axes[0]

# np.linspace(35, 165, 300) creates 300 evenly spaced x values across the
# plot range — enough points to make the line appear perfectly smooth.
x_line = np.linspace(35, 165, 300)

# Draw the guessed line: for each x value, compute ŷ = b0 + b1*x
ax.plot(
    x_line, b0_guess + b1_guess * x_line,
    'r-', linewidth=2,
    label=f'ŷ = {b0_guess} + {b1_guess}x  (guessed line)'
)

# Overlay the actual data points on top of the line (zorder=4 ensures they
# are drawn above the line so they are not hidden behind it)
ax.scatter(area, price, color='steelblue', s=45,
           edgecolors='k', linewidth=0.4, alpha=0.8, zorder=4)

# Draw one vertical red line per data point, connecting the actual price (yi)
# to the predicted price (yhi) directly above or below it on the guessed line.
# The length of each red line is the absolute residual — a long line means a
# large error; a short line means the prediction was close.
for xi, yi, yhi in zip(area, price, y_hat):
    ax.plot([xi, xi], [yi, yhi], color='tomato', linewidth=0.9, alpha=0.7)

# ax.plot([], []) with no data creates a dummy line used only to add
# a "Residuals" entry to the legend — the actual residual lines were
# drawn in the loop above and cannot be given a label individually.
ax.plot([], [], color='tomato', linewidth=1.5, label='Residuals  (r_i = y_i − ŷ_i)')

ax.set_xlabel('Floor area (m²)')
ax.set_ylabel('Sale price (£k)')
ax.set_title('Figure 3a: Residuals on a guessed line')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.25)

# ── Right panel: histogram of residuals ───────────────────────────────────────
# A histogram shows how the 60 residual values are distributed.
# For a well-fitted line, this distribution should be centred on zero and
# roughly bell-shaped — meaning errors above and below are equally common.
# If the histogram is shifted left or right, the line is systematically
# over- or under-predicting across the whole dataset.
ax = axes[1]

ax.hist(residuals, bins=20, color='tomato',
        edgecolor='white', linewidth=0.5, alpha=0.85)

# A vertical dashed line at zero helps us see whether the residuals are
# centred — if the histogram is symmetric around this line, the model
# has no systematic bias in either direction.
ax.axvline(0, color='black', linewidth=1.5, linestyle='--', label='Zero error (perfect prediction)')

ax.set_xlabel('Residual  (£k)')
ax.set_ylabel('Count')
ax.set_title('Figure 3b: Distribution of residuals\n(should be centred on zero for a good fit)')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.25)

plt.tight_layout()
plt.show()

# ── Step 5: Compute the total loss ────────────────────────────────────────────
# The sum of squared residuals (SSE) is our loss function — a single number
# that summarises how badly the guessed line fits all 60 data points.
# Squaring the residuals means large errors are penalised much more heavily
# than small ones. Our goal in the next section is to find the β₀ and β₁
# that make this number as small as possible.
sse = np.sum(residuals ** 2)
print(f'Sum of squared residuals (SSE / Loss) = {sse:.1f}')

---

### 4.4 The Least-Squares Method - Finding the Optimal Parameters

In the previous section we guessed $\beta_0 = 60$ and $\beta_1 = 1.5$. In the plot above we computed the residuals, and calculated the sum of squared residuals (our loss). The value we found was `34016.3`, which is fairly large. In other words, there is a great deal of error in this fit. The natural question is: how do we find the *best* values of $\beta_0$ and $\beta_1$, the ones that minimise this loss? To do this, we first need to define our loss function: our clearly defined method for quantifying error.

#### The loss function

The quantity we want to minimise is the **sum of squared residuals (SSE)**:

$$\text{Loss} = \text{SSE} = \sum_{i=1}^{n} \left( y_i - \hat{y}_i \right)^2 = \sum_{i=1}^{n} \left( y_i - \beta_0 - \beta_1 x_i \right)^2$$

We focus on this part as it's equivalent to the rest anyway:

$$\text{Loss} = \sum_{i=1}^{n} \left( y_i - \beta_0 - \beta_1 x_i \right)^2$$

Reading this left to right:

- $i$ is an index that counts through each data point in turn: house 1, house 2, all the way to house $n = 60$
- $y_i$ is the actual sale price of house $i$: the true value we observed
- $\beta_0 + \beta_1 x_i$ is what the model predicts for house $i$ given its floor area $x_i$: this is $\hat{y}_i$
- $y_i - \beta_0 - \beta_1 x_i$ is therefore the residual for house $i$: how far the prediction is from the truth
- $(\ldots)^2$ squares that residual
- $\sum_{i=1}^{n}$ adds up all 60 squared residuals into a single number

So the loss is simply: compute the prediction error for every house, square each error, then add them all up. A small loss means the line is close to most data points; a large loss means it is far away from many of them. We square the residuals for two reasons:

1. **Signs cancel out**: without squaring, a residual of +20 and a residual of -20 would cancel each other and give a total of zero, falsely suggesting a perfect fit. Squaring makes all contributions positive.
2. **Large errors are penalised more heavily**: a residual of 20 contributes 400 to the loss; a residual of 10 contributes only 100. This means the optimisation process is particularly motivated to eliminate large errors, which is usually the right behaviour.

#### From guessing to searching to solving

In Notebook 4 you saw two approaches to minimising a loss function: **exhaustive grid search** (trying every possible parameter value on a fixed grid) and **gradient descent** (following the slope of the loss surface downhill). Both of those approaches were necessary because the loss landscapes we examined had no simple algebraic solution, so you had to search numerically.

For linear regression with squared error, something more powerful is available. The loss function $\text{SSE}$ is a **convex** function of $\beta_0$ and $\beta_1$. Recall from Notebook 4 that convex functions have a single global minimum with no local minima to get trapped in. Furthermore, SSE is a smooth quadratic (a parabola-shaped bowl) in both parameters, which means we can find its minimum analytically using calculus rather than numerically by searching.

---

Setting the partial derivatives of SSE with respect to $\beta_0$ and $\beta_1$ to zero (exactly the condition for a minimum) and solving the resulting equations gives us the **closed-form solution**, known as **Ordinary Least Squares (OLS)**.

To understand why, we need to briefly explain two ideas. First, a **partial derivative** is simply the gradient (slope) of the loss in the direction of one specific parameter, while treating all other parameters as fixed constants. In Notebook 4 you saw that gradient descent computes the gradient with respect to $\theta$ and uses it to decide which direction to step. A partial derivative is exactly the same idea, just applied to one parameter at a time in a multi-parameter setting: $\frac{\partial \text{SSE}}{\partial \beta_0}$ measures how steeply SSE changes as $\beta_0$ moves, with $\beta_1$ held still, and vice versa.

Second, why set the derivative to zero? Recall from Notebook 4 that at the very bottom of a loss surface, the global minimum, the surface is perfectly flat in every direction. A flat surface has a slope of zero. So the minimum is precisely the point where the gradient is zero in every direction. For SSE with two parameters, this gives us two equations (one for each parameter), which we solve simultaneously to find the exact optimal $\beta_0$ and $\beta_1$ in a single calculation: no iteration, no learning rate, no risk of overshooting. This is only possible because SSE is a smooth convex bowl with a unique minimum; for more complex models the equations have no closed-form solution and we must fall back on gradient descent.

To recap, setting the partial derivatives of SSE with respect to $\beta_0$ and $\beta_1$ to zero (exactly the condition for a minimum) and solving the resulting equations gives us the **closed-form solution**, known as **Ordinary Least Squares (OLS)**:

<br>

$$\beta_1 = \frac{\sum_{i=1}^{n}(x_i - \bar{x})(y_i - \bar{y})}{\sum_{i=1}^{n}(x_i - \bar{x})^2} \qquad \beta_0 = \bar{y} - \beta_1 \bar{x}$$

Here $\bar{x}$ and $\bar{y}$ are the sample means of $x$ and $y$ respectively.

#### Understanding the formulae

These equations look dense but each component has a clear meaning:

- **Numerator of $\beta_1$**: $\sum(x_i - \bar{x})(y_i - \bar{y})$. This is the **covariance** of $x$ and $y$, measuring how much $x$ and $y$ move together. You saw this quantity before in the Pearson correlation coefficient in Notebook 3: when large $x$ values tend to occur with large $y$ values, this sum is positive, giving a positive slope; when large $x$ values occur with small $y$ values, it is negative.

- **Denominator of $\beta_1$**: $\sum(x_i - \bar{x})^2$. This is the **variance** of $x$ (up to a constant factor), measuring how spread out the $x$ values are. Dividing by this normalises the slope so it is expressed in units of $y$ per unit of $x$: £k per m² in our case.

- **$\beta_0 = \bar{y} - \beta_1 \bar{x}$**: once we know the slope, the intercept is determined by the requirement that the fitted line must pass through the point $(\bar{x}, \bar{y})$, the mean of the data. This is always true for the OLS solution, regardless of the dataset.

The closed-form solution is a significant advantage over grid search and gradient descent for this specific problem: it requires no learning rate, no iteration count, no risk of getting stuck in a local minimum, and no approximation, delivering the exact global optimum in a single calculation. This is why OLS remains the standard method for linear regression despite being over 200 years old.

That said, this only holds because the model is linear and the loss is squared error. For more complex models, such as neural networks, logistic regression, or non-linear models, no closed-form solution exists and we must fall back on gradient descent, exactly as in Notebook 4.

---

The figure below brings the loss surface concept from Notebook 4 directly into the context of linear regression. Rather than an abstract synthetic curve, the loss surface here is computed from the actual house-price data: every point on the surface represents a specific combination of $\beta_0$ and $\beta_1$, and its height (shown by colour) is the SSE that results from using those parameters.

**Figure 4a** (left) shows the scatter data with your current regression line overlaid and the residuals drawn as vertical red lines. The title shows the current SSE so you can see exactly how the loss changes as you move the sliders.

**Figure 4b** (right) shows the SSE loss surface as a contour map, where darker regions have lower loss and are therefore better. The yellow star marks the OLS optimal solution: the single point on the surface with the lowest possible SSE. The red dot marks your current position, where your chosen $\beta_0$ and $\beta_1$ sit on the loss surface.

**Try the following:**

- Start with the default values ($\beta_0 = 60$, $\beta_1 = 1.5$) and note the SSE and the position of the red dot on the loss surface. It sits far from the yellow star.
- Drag the sliders and watch both panels update simultaneously. As you move the red dot toward the yellow star in Figure 4b, observe how the residuals in Figure 4a shrink and the SSE falls.
- Try to manually find the minimum by dragging the sliders until the red dot is as close to the yellow star as you can get. Note the $\beta_0$ and $\beta_1$ values you land on.
- Click **Show OLS optimum** to jump instantly to the analytically computed best values and see exactly where the global minimum is. Compare these values to your manual estimate.
- Click **Reset** to return to the initial guess and start again.

The loss surface for linear regression is a smooth convex bowl. There is one minimum, it has no local minima to get trapped in, and the OLS formula finds it exactly in a single calculation rather than requiring an iterative search.

In [ ]:
# Listing 6.
%matplotlib widget
from visualisations.Figure_4 import show
show()

The code below implements the OLS closed-form formulae directly in NumPy, computing $\beta_0$ and $\beta_1$ step by step from the data rather than using a library function. This is intentional: seeing each component of the formula computed explicitly makes the connection between the mathematics and the code concrete.

The calculation proceeds in four steps:

1. **Compute the means**: $\bar{x}$ (mean floor area) and $\bar{y}$ (mean sale price) are computed using `.mean()`. These are the anchor points for both formulae.
2. **Compute $\beta_1$**: for each data point, subtract the mean from both $x$ and $y$, multiply the two deviations together, and sum across all points. This gives the numerator (the covariance of $x$ and $y$). Divide by the sum of squared $x$ deviations (the variance of $x$) to get the slope.
3. **Compute $\beta_0$**: once $\beta_1$ is known, the intercept follows immediately from $\beta_0 = \bar{y} - \beta_1 \bar{x}$, which ensures the fitted line passes through the point $(\bar{x}, \bar{y})$.
4. **Make an example prediction**: the fitted model is applied to a new house with floor area 100 m² to show how the parameters are used in practice.

The output also shows that the values of $\beta_0$ and $\beta_1$ recovered here should be close to the true values used to generate the data ($\beta_0 = 50$, $\beta_1 = 1.8$), but not exactly equal, because the noise added during data generation means the sample estimates will always differ slightly from the true population values.


In [ ]:
# Listing 7.
# Derive the OLS coefficients step by step from the closed-form formulae

x_bar = area.mean()    # x̄ — mean of the predictor
y_bar = price.mean()   # ȳ — mean of the response

# beta_1 = sum[(x_i - x_bar)(y_i - y_bar)] / sum[(x_i - x_bar)^2]
#        = Cov(x, y) / Var(x)
beta_1 = np.sum((area - x_bar) * (price - y_bar)) / np.sum((area - x_bar)**2)

# beta_0 = y_bar - beta_1 * x_bar  (ensures the line passes through (x_bar, y_bar))
beta_0 = y_bar - beta_1 * x_bar

print('Least-squares closed-form solution')
print('-' * 40)
print(f'  x̄ (mean area)  = {x_bar:.2f} m²')
print(f'  ȳ (mean price) = {y_bar:.2f} £k')
print(f'  β₁             = {beta_1:.4f}')
print(f'  β₀             = {beta_0:.4f}')
print()
print(f'  Fitted model: ŷ = {beta_0:.2f} + {beta_1:.2f} · x')
print()
print('Example prediction:')
print(f'  A 100 m² house is predicted to sell for:')
print(f'  ŷ = {beta_0:.2f} + {beta_1:.2f} × 100 = {beta_0 + beta_1 * 100:.2f} £k')

---

### 4.5 Evaluating the Fit

Fitting the line is only half the job. Once we have estimated $\beta_0$ and $\beta_1$, we need to ask: **how good is this model?** More specifically, how well will it predict the sale price of houses it has never seen before?

This question has two parts that are easy to confuse:

- **How well does the model fit the training data?** This is straightforward to measure.
- **How well will the model generalise to new, unseen data?** This requires a different approach.

#### Why we need a test set

A model that has been trained on a dataset will always appear to perform well on that same dataset, because it has, in a sense, already seen the answers. If we use the training data to evaluate the model, we are asking "how well does the model remember the data it was trained on?" rather than "how well does it predict new data?". These are very different questions.

The standard solution is to split the available data into two parts before training:

- **Training set**: the data used to fit the model (estimate $\beta_0$ and $\beta_1$)
- **Test set**: a held-out portion of the data that the model never sees during training, used only for evaluation

By measuring performance on the test set, we get an honest estimate of how well the model will generalise: its performance on data it has genuinely never encountered. A model that performs well on training data but poorly on test data is said to be **overfitting**, meaning it has learned the noise in the training data rather than the underlying pattern. You will study overfitting in depth in later notebooks; for now, the key habit to build is always evaluating on held-out data.

#### Evaluation metrics for regression

Once we have a test set, we can compute predictions for every test point and compare them to the true values. Several metrics are commonly used, each capturing a slightly different aspect of model quality:

**Mean Absolute Error (MAE)**

$$\text{MAE} = \frac{1}{n} \sum_{i=1}^{n} |y_i - \hat{y}_i|$$

MAE is the average absolute residual: the average gap between predictions and true values, ignoring whether the error was positive or negative. It is expressed in the same units as $y$ (£k in our case), making it directly interpretable: an MAE of 15 means the model is wrong by £15k on average. MAE treats all errors equally regardless of their size.

**Root Mean Squared Error (RMSE)**

$$\text{RMSE} = \sqrt{\frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2}$$

RMSE is the square root of the average squared residual. Like MAE it is in the same units as $y$, but because errors are squared before averaging, large errors are penalised much more heavily than small ones. A model with one catastrophically wrong prediction will have a much higher RMSE than MAE. RMSE is more sensitive to outliers and is the natural companion to the SSE loss function used during training.

**$R^2$: the Coefficient of Determination**

$$R^2 = 1 - \frac{\sum_{i=1}^{n}(y_i - \hat{y}_i)^2}{\sum_{i=1}^{n}(y_i - \bar{y})^2}$$

$R^2$ measures the proportion of the total variation in $y$ that is explained by the model. The denominator is the total variance in the data: how spread out the $y$ values are around their mean. The numerator is the residual variance: how much variation remains unexplained after fitting. Subtracting the ratio from 1 gives a score between 0 and 1 (in well-behaved cases):

- $R^2 = 1$ means the model explains all variation perfectly: every prediction is exact
- $R^2 = 0$ means the model explains nothing: it is no better than simply predicting the mean $\bar{y}$ for every house
- $R^2 = 0.85$ means the model accounts for 85% of the variation in sale prices

Unlike MAE and RMSE, $R^2$ is unitless and always on the same scale, making it easy to compare models trained on different datasets.

#### Choosing between metrics

There is no single "best" metric. The right choice depends on what matters in your application:

- Use **MAE** when you want an easily interpretable average error in original units and large errors are not disproportionately costly
- Use **RMSE** when large errors are particularly undesirable and you want the metric to reflect that
- Use **$R^2$** when you want a relative measure of how much of the outcome the model explains, or when comparing models on different datasets

In practice it is common to report all three. A good model should have low MAE and RMSE and a high $R^2$, but always measured on the test set, not the training set.

---


### 4.6 Linear Regression Example

So far we have been working with a small dataset of 60 houses, which was sufficient to illustrate the mechanics of fitting a line and computing residuals. For a proper evaluation of how well the model generalises to unseen data, we need more observations: a small dataset produces noisy, unreliable test metrics because the test set itself is too small to be representative.

To do this properly we will use **scikit-learn** (`sklearn`), the most widely used machine learning library in Python. Scikit-learn provides efficient, well-tested implementations of hundreds of machine learning algorithms, from linear regression to neural networks, all following a consistent interface. It also provides essential utilities for data preparation, model evaluation, and cross-validation. You will rely on it throughout the rest of these notebooks, so this section also serves as your introduction to how it works.

The key things to know about scikit-learn at this stage are:

- Every model in scikit-learn follows the same three-step pattern: **create**, **fit**, **predict**, regardless of whether you are fitting a linear regression, a decision tree, or a neural network
- Models are always fitted on training data only and evaluated on held-out test data
- The library handles the underlying mathematics for you, but understanding what it is doing under the hood, as you have done in the previous sections, is essential for using it correctly and interpreting its outputs

---

The code below generates a larger synthetic dataset of 500 house sales using the same underlying relationship ($\text{price} = 1.8 \times \text{area} + 50 + \varepsilon$) but with a slightly larger noise standard deviation of 22 £k to make the problem a little more realistic. The data is stored in a **Pandas DataFrame**, a two-column table with one row per house, and summarised using `.head()` and `.describe()`.

Using a DataFrame here is good practice for two reasons. First, it keeps the feature ($x$) and target ($y$) together in a named, structured format that is easier to inspect than raw NumPy arrays. Second, scikit-learn, the machine learning library we will use to fit the model, accepts DataFrames directly, so there is no conversion needed. The `.describe()` output gives a quick sanity check: the floor area should range from roughly 30 to 200 m², and the sale prices should be broadly centred around the values you would expect from the true relationship.

In [ ]:
# Listing 8.
# ── Generate a larger synthetic housing dataset ───────────────────────────────
# We use a new, larger dataset here (500 samples instead of 60) to give the
# train/test split more data to work with — a 20% test set will give us
# 100 unseen houses to evaluate on, which is enough for reliable metrics.
rng_split = np.random.default_rng(7)   # separate seed so this dataset is independent
n_large   = 500

# Floor area: uniformly distributed between 30 and 200 m²
area_large  = rng_split.uniform(30, 200, n_large)

# Sale price: same true relationship as before (β₀=50, β₁=1.8) but with more noise
# to make the problem a little harder and the train/test comparison more interesting
price_large = 1.8 * area_large + 50 + rng_split.normal(0, 22, n_large)

# ── Store in a Pandas DataFrame ───────────────────────────────────────────────
# Keeping the data in a DataFrame makes it easy to inspect, summarise, and
# pass to scikit-learn — and gets us into the habit of working with structured data.
df = pd.DataFrame({
    'floor_area_m2': area_large,    # input feature (x)
    'sale_price_kgbp': price_large, # target variable (y)
})

print(f'Dataset shape: {df.shape[0]} rows × {df.shape[1]} columns')
print()
print('First 5 rows:')
print(df.head())
print()
print('Summary statistics:')
print(df.describe().round(2))

---

With the dataset created and stored in a DataFrame, the next step is to divide it into a **training set** and a **test set** before fitting any model. This split must happen *before* training: if the model sees the test data at any point during fitting, the evaluation will be optimistic and misleading.

We use scikit-learn's `train_test_split()` function from the `sklearn.model_selection` module to do this. It takes care of two things simultaneously: shuffling the data randomly (so the split is not accidentally ordered by floor area or price) and then dividing it into the two portions.

```python
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
```

The key parameters are:

| Parameter | Value | Meaning |
|---|---|---|
| `X` | feature DataFrame | The input features: floor area in our case |
| `y` | target Series | The output we are trying to predict: sale price |
| `test_size` | `0.2` | Reserve 20% of the data for testing; 80% goes to training |
| `random_state` | `42` | Fixes the random shuffle so the split is the same every time the cell is run, essential for reproducibility |

The function returns **four objects** in this order:

- `X_train`: the training features (400 rows, 1 column)
- `X_test`: the test features (100 rows, 1 column), held back from training entirely
- `y_train`: the training target values (400 prices)
- `y_test`: the test target values (100 prices), used only for evaluation

Notice that `X` uses double brackets `df[['floor_area_m2']]` rather than single brackets. This keeps it as a **DataFrame** with shape (500, 1) rather than collapsing it to a one-dimensional **Series**. Scikit-learn's `LinearRegression` expects a 2D feature matrix, one row per observation and one column per feature, so this convention matters and you will see it throughout these notebooks.

The code also prints summary statistics for both the training and test sets after splitting. This is a useful sanity check: if the two sets have very similar means and standard deviations for both the feature and the target, the split is representative, meaning neither set is accidentally biased toward large houses or high prices. If the means were very different it would suggest the data was not shuffled properly, and our test metrics would not be trustworthy.

Read the code and when ready, execute it.

In [ ]:
# Listing 9.
# ── Split the dataset into training and test sets ─────────────────────────────
# We use scikit-learn's train_test_split() function, which shuffles the data
# randomly and then divides it into two portions.
#
# Parameters:
#   test_size=0.2   — reserve 20% of the data (100 houses) for testing;
#                     the remaining 80% (400 houses) are used for training
#   random_state=42 — fixes the random shuffle so the split is reproducible;
#                     without this, every run would produce a different split

X = df[['floor_area_m2']]    # feature matrix — double brackets keep it as a DataFrame
                              # scikit-learn expects shape (n_samples, n_features)
y = df['sale_price_kgbp']    # target vector — single brackets returns a Series

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
)

# ── Inspect the split ─────────────────────────────────────────────────────────
print(f'Total samples:     {len(df)}')
print(f'Training samples:  {len(X_train)}  ({len(X_train)/len(df):.0%} of total)')
print(f'Test samples:      {len(X_test)}   ({len(X_test)/len(df):.0%} of total)')
print()

# Confirm the training and test sets have similar distributions —
# if one set were systematically different, our evaluation would be biased.
# The means and standard deviations should be close between the two sets.
print('Feature (floor area) statistics:')
print(f'  Training mean: {X_train["floor_area_m2"].mean():.1f} m²   '
      f'std: {X_train["floor_area_m2"].std():.1f} m²')
print(f'  Test mean:     {X_test["floor_area_m2"].mean():.1f} m²   '
      f'std: {X_test["floor_area_m2"].std():.1f} m²')
print()
print('Target (sale price) statistics:')
print(f'  Training mean: £{y_train.mean():.1f}k   std: £{y_train.std():.1f}k')
print(f'  Test mean:     £{y_test.mean():.1f}k   std: £{y_test.std():.1f}k')

---

With the training and test sets prepared, we can now fit the model and evaluate it. We use scikit-learn's `LinearRegression` class from the `sklearn.linear_model` module, which implements OLS under the hood: the same closed-form formulae we derived manually in section 4.4, but optimised and production-ready.

Scikit-learn follows a consistent three-step pattern for every model it provides, and you will see this same pattern throughout the remaining notebooks:

```python
from sklearn.linear_model import LinearRegression

model = LinearRegression()   # Step 1: create the model object
model.fit(X_train, y_train)  # Step 2: train it on the training data
y_pred = model.predict(X)    # Step 3: use it to make predictions
```

**Step 1 — `LinearRegression()`** creates an unfitted model object. At this point the model has no parameter values; $\beta_0$ and $\beta_1$ are unknown. Think of this as buying a blank form that is ready to be filled in.

**Step 2 — `.fit(X_train, y_train)`** trains the model by finding the OLS optimal $\beta_0$ and $\beta_1$ from the training data. After this call the parameters are fixed and stored inside the model object as:
- `model.coef_[0]`: the fitted slope $\beta_1$
- `model.intercept_`: the fitted intercept $\beta_0$

Note that `fit()` receives only the **training data**, `X_train` and `y_train`. The test set is kept completely hidden at this stage.

**Step 3 — `.predict(X)`** applies the fitted model to a feature matrix and returns predicted $\hat{y}$ values. We call it twice: once on `X_train` to get in-sample predictions, and once on `X_test` to get out-of-sample predictions on data the model has never seen.

The code then defines four evaluation metric functions from scratch using NumPy (MSE, RMSE, MAE, and $R^2$) and reports all four for both the training and test sets side by side. Presenting them in a table makes it easy to spot the key comparison: if the training and test metrics are similar, the model is generalising well to unseen data. If training metrics are noticeably better than test metrics, the model has overfitted, meaning it has learned patterns specific to the training data that do not hold in general. For a simple linear regression model on clean data, the two sets of metrics should be close.

In [ ]:
# Listing 10.
# ── Fit the model on the training set ────────────────────────────────────────
# LinearRegression() creates an unfitted model object — at this point it has
# no parameter values; β₀ and β₁ are unknown.
# .fit(X_train, y_train) finds the OLS optimal β₀ and β₁ by minimising the
# sum of squared residuals on the training data. After this call the model
# is fully trained and ready to make predictions.
model = LinearRegression()
model.fit(X_train, y_train)

# scikit-learn stores the fitted parameters as attributes on the model object:
#   model.coef_[0]   = β₁ (slope)
#   model.intercept_ = β₀ (intercept)
print(f'Fitted parameters:')
print(f'  β₀ (intercept) = {model.intercept_:.4f}')
print(f'  β₁ (slope)     = {model.coef_[0]:.4f}')
print(f'  True values:   β₀ = 50, β₁ = 1.8')
print()

# ── Generate predictions ──────────────────────────────────────────────────────
# .predict() applies the fitted model to a feature matrix and returns ŷ values.
# We generate predictions for both sets so we can compare training vs test performance.
# Crucially, y_pred_test is computed on data the model has never seen — this is
# our honest estimate of how well the model generalises.
y_pred_train = model.predict(X_train)
y_pred_test  = model.predict(X_test)

# ── Evaluation metric functions ───────────────────────────────────────────────
# We implement these from scratch using NumPy so the formulae are transparent.
# In practice you could also use sklearn.metrics.mean_squared_error etc.,
# but writing them by hand reinforces what each metric is actually computing.

def mse(y_true, y_pred):
    """Mean Squared Error — average of squared residuals. Units: £k²"""
    return np.mean((y_true - y_pred) ** 2)

def rmse(y_true, y_pred):
    """Root Mean Squared Error — square root of MSE. Units: £k (same as y)"""
    return np.sqrt(mse(y_true, y_pred))

def mae(y_true, y_pred):
    """Mean Absolute Error — average of absolute residuals. Units: £k"""
    return np.mean(np.abs(y_true - y_pred))

def r2(y_true, y_pred):
    """
    R-squared — proportion of variance in y explained by the model.
    ss_res: how much variance remains after fitting (residual sum of squares)
    ss_tot: how much variance exists in total (total sum of squares)
    R² = 1 means perfect fit; R² = 0 means the model is no better than
    predicting the mean for every observation.
    """
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - y_true.mean()) ** 2)
    return 1 - ss_res / ss_tot

# ── Report metrics for both sets ──────────────────────────────────────────────
# Comparing training and test metrics tells us whether the model generalises:
# - If both are similar, the model is generalising well
# - If training metrics are much better than test metrics, the model is overfitting
print('─' * 45)
print(f'{"Metric":<10} {"Training":>15} {"Test":>15}')
print('─' * 45)
print(f'{"MSE":<10} {mse(y_train, y_pred_train):>14.3f}  {mse(y_test, y_pred_test):>14.3f}  £k²')
print(f'{"RMSE":<10} {rmse(y_train, y_pred_train):>14.3f}  {rmse(y_test, y_pred_test):>14.3f}  £k')
print(f'{"MAE":<10} {mae(y_train, y_pred_train):>14.3f}  {mae(y_test, y_pred_test):>14.3f}  £k')
print(f'{"R²":<10} {r2(y_train, y_pred_train):>14.4f}  {r2(y_test, y_pred_test):>14.4f}')
print('─' * 45)

---

Two standard diagnostic plots are produced using the code below, to evaluate the fitted model visually, complementing the numerical metrics reported above.

The **left panel** shows the scatter data with the fitted regression line overlaid. Training points are shown in blue and test points in red, so you can see at a glance that the line was fitted only to the blue points; the red test points are genuinely unseen. A good fit should show the line running through the middle of both clouds with roughly equal scatter above and below. The title reports the $R^2$ for both sets so you can immediately compare in-sample and out-of-sample performance.

The **right panel** shows the distribution of residuals for both the training and test sets as overlapping histograms. For a well-fitted linear regression model, the residual distribution should be:

- **Centred on zero**: if the histogram is shifted left or right, the model is systematically over- or under-predicting across the whole dataset, which suggests the model may be missing something important
- **Roughly bell-shaped and symmetric**: asymmetry or a long tail in one direction suggests the errors are not random, which can indicate that a linear model is not the right choice for this data
- **Similar for training and test**: if the two histograms are noticeably different, the model may be overfitting, with the training residuals smaller and tighter while the test residuals are wider and more spread out

The RMSE values in the title give a quick numerical summary of the spread of each histogram. Roughly speaking, RMSE tells you the typical size of the errors in £k.

In [ ]:
# Listing 11.
# ── Visualise the fitted model and residual distribution ──────────────────────
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
fig.canvas.header_visible = False
fig.canvas.resizable = True
fig.canvas.toolbar_visible = True
fig.canvas.toolbar_position = 'right'
# ── Left panel: scatter with fitted line ──────────────────────────────────────
ax = axes[0]

# Plot training and test points in different colours so students can see
# which observations were used for fitting and which were held back
ax.scatter(X_train, y_train, color='steelblue', s=35, alpha=0.6,
           edgecolors='k', lw=0.3, label=f'Training data (n={len(X_train)})', zorder=3)
ax.scatter(X_test, y_test, color='tomato', s=35, alpha=0.8,
           edgecolors='k', lw=0.3, label=f'Test data (n={len(X_test)})', zorder=4)

# Draw the fitted line across the full range of floor areas.
# We create x_range as a DataFrame with the same column name as X_train
# to avoid a scikit-learn warning about mismatched feature names.
x_range = pd.DataFrame(
    np.linspace(X['floor_area_m2'].min() - 5,
                X['floor_area_m2'].max() + 5, 300),
    columns=['floor_area_m2']
)
ax.plot(x_range, model.predict(x_range), color='black', linewidth=2.5, zorder=2,
        label=f'Fitted line: ŷ = {model.intercept_:.1f} + {model.coef_[0]:.2f}x')

ax.set_xlabel('Floor area (m²)', fontsize=11)
ax.set_ylabel('Sale price (£k)', fontsize=11)
ax.set_title(
    f'Fitted regression line\n'
    f'Train R² = {r2(y_train, y_pred_train):.3f}   '
    f'Test R² = {r2(y_test, y_pred_test):.3f}',
    fontsize=10,
)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.2)

# ── Right panel: residual distribution ────────────────────────────────────────
# We plot residuals for both training and test sets as overlapping histograms.
# For a well-fitted model both distributions should be:
#   - centred on zero (no systematic over- or under-prediction)
#   - roughly bell-shaped (errors are random, not structured)
# A noticeable shift between training and test residuals can indicate overfitting.
ax = axes[1]

train_residuals = y_train - y_pred_train
test_residuals  = y_test  - y_pred_test

ax.hist(train_residuals, bins=25, color='steelblue', alpha=0.6,
        edgecolor='white', lw=0.4, label='Training residuals')
ax.hist(test_residuals,  bins=25, color='tomato', alpha=0.6,
        edgecolor='white', lw=0.4, label='Test residuals')

# Vertical dashed line at zero — both histograms should be symmetric around this
ax.axvline(0, color='black', linewidth=1.8, linestyle='--', label='Zero error')

ax.set_xlabel('Residual (£k)', fontsize=11)
ax.set_ylabel('Count', fontsize=11)
ax.set_title(
    f'Residual distributions\n'
    f'Train RMSE = {rmse(y_train, y_pred_train):.1f} £k   '
    f'Test RMSE = {rmse(y_test, y_pred_test):.1f} £k',
    fontsize=10,
)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.2)

plt.suptitle('Figure 5: Model fit and residual distribution — training vs test',
             fontsize=11)
plt.tight_layout(rect=[0, 0, 1, 0.93])
plt.show()

The fitted model appears to work well on data it has never seen before.

---

### 4.7 The Four Key Assumptions of Linear Regression

The least-squares method is only guaranteed to produce the **Best Linear Unbiased Estimates (BLUE)** when four conditions hold, known collectively as the **Gauss–Markov assumptions**. Violating them does not necessarily make the model useless, but it does mean that standard error estimates and inferential tests cannot be trusted.

1. **Linearity**: the true relationship between $x$ and $y$ is linear. *Check:* a plot of residuals vs fitted values should show no systematic curve.

2. **Independence**: observations are independent of one another. *Check:* residuals plotted against observation index should show no trend or autocorrelation.

3. **Homoscedasticity**: residuals have constant variance across all fitted values. *Check:* the spread of $|r_i|$ should be roughly flat across the range of $\hat{y}$.

4. **Normality**: residuals are normally distributed. *Check:* a histogram of residuals should follow a bell curve.

The cell below produces all four diagnostic plots using the training residuals.


In [ ]:
# Listing 12.
%matplotlib widget
from visualisations.Figure_6 import show
show(model,X_train,y_train) # Uses data we created in previous steps.


---


## 5. Multiple Linear Regression

🔙 [Back to Table of Contents](#Table-of-Contents)

### 5.1 The Model

Real datasets rarely have just one predictor. A house's sale price is not determined solely by its floor area: the number of bedrooms, the distance to the nearest train station, and the age of the property all contribute. **Multiple Linear Regression (MLR)** extends the simple model to $p$ features:

$$y = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + \cdots + \beta_p x_p$$

The structure is identical to simple linear regression in that the model is still a straight line, but now that line exists in a higher-dimensional space. With two predictors, the model describes a flat plane through the data; with three or more predictors, it describes a hyperplane that we cannot visualise directly but can still fit and interpret mathematically.

Each coefficient $\beta_j$ in the model represents the **marginal contribution** of feature $x_j$: it quantifies how much $y$ changes for a one-unit increase in $x_j$, *holding all other features constant*. This "holding all other features constant" clause is important and easy to overlook. In simple regression, $\beta_1$ captures everything the slope can, since there is only one predictor. In multiple regression, each $\beta_j$ tells you the effect of that specific feature after accounting for all the others. This means the coefficient of floor area in a multiple regression model (which also includes number of bedrooms) will generally be different from the coefficient of floor area in a simple regression model, because some of what appeared to be the "effect" of floor area was actually the effect of the number of bedrooms, which is correlated with floor area.

$\beta_0$ remains the intercept: the predicted $y$ when all features are simultaneously zero. In practice this value is often not directly interpretable (a house with zero floor area, zero bedrooms, and zero distance to a station does not exist), but it is a necessary part of the model to ensure the line can be positioned correctly in space.

The training loss is the same sum-of-squared-residuals objective, now expanded over all $p$ predictors:

$$\text{Loss} = \sum_{i=1}^{n} \left( y_i - (\beta_0 + \beta_1 x_{i1} + \beta_2 x_{i2} + \cdots + \beta_p x_{ip}) \right)^2$$

The loss function has exactly the same form as before: we are still summing squared differences between true and predicted values across all $n$ observations. What changes is that there are now $p + 1$ parameters ($\beta_0, \beta_1, \ldots, \beta_p$) to optimise simultaneously rather than just two.

The OLS closed-form solution generalises naturally to multiple predictors using matrix algebra. The mathematics becomes more compact but the core idea is unchanged: set the gradient of the loss to zero and solve. We will not derive this here, but it is worth knowing that scikit-learn's `LinearRegression` uses exactly this approach internally, computing the exact optimal solution in a single operation regardless of the number of features. For now, we let scikit-learn handle the optimisation and focus on building and interpreting multiple regression models.

#### Synthetic Dataset: Multi-Feature House Prices

The cell below generates a synthetic house price dataset with three features. The dataset parameters are summarised in the table below.

| Variable | Type | Range / Distribution | Notes |
|----------|------|----------------------|-------|
| `area` | Continuous (float) | Uniform, 40–200 m² | Floor area |
| `bedrooms` | Discrete (int) | Chosen from {1, 2, 3, 4, 5} | Number of bedrooms |
| `distance` | Continuous (float) | Uniform, 1–40 km | Distance from city centre |
| `price` | Continuous (float) | Derived + noise ($\sigma=20$) | True model: $30 + 1.5 \times \text{area} + 12 \times \text{bedrooms} - 2 \times \text{distance}$ |

The true coefficients are $\beta_1 = 1.5$, $\beta_2 = 12$, and $\beta_3 = -2.0$. A well-fitted model should recover values close to these.

#### A new dataset with multiple variables

To demonstrate multiple linear regression we need a dataset with more than one input feature. The code in the cell below generates a synthetic dataset of 200 house sales with three predictors:

- **Floor area** (m²): a continuous feature drawn uniformly between 40 and 200 m²
- **Number of bedrooms**: a discrete feature taking integer values from 1 to 5, chosen at random
- **Distance to nearest train station** (km): a continuous feature drawn uniformly between 1 and 40 km

The true underlying model used to generate the prices is:

$$\text{price} = 30 + 1.5 \times \text{area} + 12.0 \times \text{bedrooms} - 2.0 \times \text{distance} + \varepsilon$$

This means:
- Every additional m² of floor area adds £1.5k to the price
- Every additional bedroom adds £12k to the price
- Every additional km from the station *reduces* the price by £2k, since proximity is valuable
- The intercept of £30k represents the base price when all features are zero

Notice that the distance coefficient is **negative**: this is the first time we have used a predictor with a negative effect. This is entirely natural in multiple regression: some features increase the target variable, others decrease it, and the sign of each coefficient captures this direction.

As before, Gaussian noise with standard deviation 20 £k is added to each observation to simulate the real-world variation in house prices that our three features cannot fully explain. The true parameter values are chosen to be realistic, and once we fit the model we should recover estimates close to these values, though not identical, because of the noise.

The data is stored in a Pandas DataFrame with one column per variable. The `.describe()` output gives a quick summary of each column's range and distribution, worth checking before fitting to confirm the data was generated as expected.


In [ ]:
# Listing 13.
# ── Step 1: Set up the random number generator ────────────────────────────────
# A separate seed (7) keeps this dataset independent from the one used in
# the simple regression example above.
rng2 = np.random.default_rng(7)
n2   = 200   # 200 house sales — enough for a reliable three-feature fit

# ── Step 2: Generate the three input features ─────────────────────────────────

# Feature 1: floor area in m² — uniformly distributed between 40 and 200 m²
area2 = rng2.uniform(40, 200, n2)

# Feature 2: number of bedrooms — a discrete choice from {1, 2, 3, 4, 5}.
# rng2.choice([1,2,3,4,5], n2) picks one value at random from the list for
# each of the 200 houses. .astype(float) converts integers to floats so
# scikit-learn can treat this column consistently with the other features.
bedrooms = rng2.choice([1, 2, 3, 4, 5], n2).astype(float)

# Feature 3: distance to the nearest train station in km — uniformly
# distributed between 1 and 40 km. Note that this feature has a *negative*
# effect on price — houses further from the station are cheaper.
distance = rng2.uniform(1, 40, n2)

# ── Step 3: Compute the target variable ──────────────────────────────────────
# The true model we use to generate prices is:
#   price = 30 + 1.5×area + 12×bedrooms − 2×distance + ε
# Each term contributes independently — this is the additive structure of
# multiple linear regression. After fitting, we should recover coefficients
# close to these true values (but not identical, because of the noise).
price2 = (
    30                           # β₀: base price (intercept)
    + 1.5  * area2               # β₁: £1.5k per additional m²
    + 12.0 * bedrooms            # β₂: £12k per additional bedroom
    - 2.0  * distance            # β₃: −£2k per additional km from station
    + rng2.normal(0, 20, n2)     # ε:  Gaussian noise with std = 20 £k
)

# ── Step 4: Store in a Pandas DataFrame ──────────────────────────────────────
# Using a DataFrame with named columns makes the data easy to inspect,
# ensures the feature names are preserved when we pass it to scikit-learn,
# and gets us into the habit of working with structured, labelled data.
df2 = pd.DataFrame({
    'area':     area2,
    'bedrooms': bedrooms,
    'distance': distance,
    'price':    price2,
})

# ── Step 5: Inspect the dataset ───────────────────────────────────────────────
# Always check the shape and summary statistics before fitting — this confirms
# the data was generated as expected and helps spot any obvious issues.
print('Dataset shape:', df2.shape)
print()
print(df2.describe().round(2))

---

### 5.2 Fitting and Interpreting a Multiple Regression Model

With the dataset prepared, fitting a multiple regression model in scikit-learn requires almost identical code to simple regression. The only difference is that the feature matrix `X` now has three columns instead of one. Scikit-learn's `LinearRegression` handles any number of features automatically, finding the OLS optimal values for all $p + 1$ coefficients simultaneously.

The code below follows the same three-step pattern introduced in section 4.6:

**Step 1 — Prepare the feature matrix and target vector.** We select the three feature columns from the DataFrame and extract them as NumPy arrays. The feature matrix `X_mlr` has shape `(200, 3)`: 200 rows (one per house) and 3 columns (one per feature). The target vector `y_mlr` has shape `(200,)`. Note that here we use `.values` to convert the DataFrame columns to plain NumPy arrays, which avoids the feature name warning we encountered earlier when mixing DataFrames and NumPy arrays.

**Step 2 — Split into training and test sets.** We use `train_test_split()` with `test_size=0.2` and `random_state=42` as before, reserving 20% of the data (40 houses) for evaluation.

**Step 3 — Fit the model.** `mlr.fit(X_tr_mlr, y_tr_mlr)` finds the values of $\beta_0, \beta_1, \beta_2, \beta_3$ that minimise the sum of squared residuals on the training data. After fitting, the learned coefficients are stored as:
- `mlr.intercept_`: the fitted $\beta_0$
- `mlr.coef_`: a NumPy array containing $[\beta_1, \beta_2, \beta_3]$, one value per feature in the same order as the columns of `X_mlr`

**Step 4 — Generate predictions and evaluate.** We call `.predict()` on both the training and test sets and compute the training SSR, test MSE, test RMSE, and test $R^2$ using the same NumPy formulas as before.

**Interpreting the coefficients** is the most important step. Each coefficient tells you the effect of a one-unit change in that feature *holding all other features constant*. A key sanity check is whether the fitted coefficients are close to the true values we used to generate the data: if they are, the model has successfully recovered the underlying relationship. The signs matter too: a positive coefficient means the feature pushes the price up; a negative coefficient means it pushes it down. We know from the data-generating process that distance should have a negative coefficient (further from the station means cheaper), so if the model recovers a positive coefficient for distance, something would have gone wrong.

The code also prints a plain-English interpretation of each coefficient, translating the numerical value into a statement about how a one-unit change in each feature affects the predicted price.


In [ ]:
# Listing 14.
# ── Step 1: Prepare the feature matrix and target vector ──────────────────────
# We select the three predictor columns by name and extract them as NumPy
# arrays using .values. This gives scikit-learn a plain 2D array with no
# column name metadata, avoiding the feature-name warnings we saw earlier.
feature_cols = ['area', 'bedrooms', 'distance']

X_mlr = df2[feature_cols].values   # shape (200, 3) — 200 rows, 3 feature columns
y_mlr = df2['price'].values         # shape (200,)  — one target value per house

# ── Step 2: Train/test split ──────────────────────────────────────────────────
# Same 80/20 split as before. random_state=42 ensures the split is identical
# every time the cell is run, so results are reproducible.
X_tr_mlr, X_te_mlr, y_tr_mlr, y_te_mlr = train_test_split(
    X_mlr, y_mlr, test_size=0.2, random_state=42
)

# ── Step 3: Fit the model ─────────────────────────────────────────────────────
# LinearRegression().fit() finds the values of β₀, β₁, β₂, β₃ that minimise
# the sum of squared residuals across all 160 training observations.
# With three features, scikit-learn solves a 4-parameter OLS problem internally.
mlr = LinearRegression()
mlr.fit(X_tr_mlr, y_tr_mlr)

# After fitting, the learned parameters are stored as:
#   mlr.intercept_  — β₀ (a single float)
#   mlr.coef_       — [β₁, β₂, β₃] (a NumPy array, one value per feature,
#                      in the same column order as X_mlr)

# ── Step 4: Generate predictions ─────────────────────────────────────────────
# .predict() applies the fitted model to compute ŷ for every observation.
# We compute predictions on both sets so we can compare in-sample and
# out-of-sample performance.
y_hat_tr_mlr = mlr.predict(X_tr_mlr)   # predictions on training data
y_hat_te_mlr = mlr.predict(X_te_mlr)   # predictions on test data (unseen during training)

# ── Step 5: Compute evaluation metrics ───────────────────────────────────────
# Training loss: sum of squared residuals — the quantity that was minimised
loss_train = np.sum((y_tr_mlr - y_hat_tr_mlr) ** 2)

# Test metrics: computed on the held-out set for an honest evaluation
mse_test = np.mean((y_te_mlr - y_hat_te_mlr) ** 2)

# R² on the test set: proportion of variance in test prices explained by the model
r2_test = 1 - (
    np.sum((y_te_mlr - y_hat_te_mlr) ** 2)
    / np.sum((y_te_mlr - y_te_mlr.mean()) ** 2)
)

# ── Step 6: Print the fitted coefficients and compare to true values ──────────
# The true values are what we used to generate the data — the model should
# recover estimates close to these, though not identical due to noise.
print('Multiple Linear Regression — fitted coefficients')
print('-' * 50)
print(f'  β₀ (intercept)  = {mlr.intercept_:.3f}  (true: 30.0)')

true_coefs = {'area': 1.5, 'bedrooms': 12.0, 'distance': -2.0}
for name, coef in zip(feature_cols, mlr.coef_):
    print(f'  β  ({name:<10}) = {coef:.4f}  (true: {true_coefs[name]})')

print()
print(f'Training Loss (SSR)  = {loss_train:.1f}')
print(f'Test MSE             = {mse_test:.3f}  £k²')
print(f'Test RMSE            = {np.sqrt(mse_test):.3f}  £k')
print(f'Test R²              = {r2_test:.4f}')

# ── Step 7: Plain-English interpretation of each coefficient ──────────────────
# Each β tells us the effect of a one-unit increase in that feature on the
# predicted price, holding all other features constant.
# The direction (increases/decreases) is determined by the sign of the coefficient.
print()
print('Interpretation of coefficients:')
for name, coef in zip(feature_cols, mlr.coef_):
    direction = 'increases' if coef > 0 else 'decreases'
    print(f'  Each 1-unit increase in {name:<10} {direction} price by £{abs(coef):.2f}k')

---

The output confirms that the model has successfully recovered the underlying relationships in the data. Working through each result:

**Fitted coefficients vs true values**

The estimated coefficients are very close to the true values used to generate the data, which is exactly what we would hope to see:

- $\hat{\beta}_1 = 1.55$ vs true $\beta_1 = 1.5$: the model correctly identifies that floor area has a positive effect on price, and the magnitude is almost exactly right
- $\hat{\beta}_2 = 12.00$ vs true $\beta_2 = 12.0$: the bedroom coefficient is recovered almost perfectly
- $\hat{\beta}_3 = -1.94$ vs true $\beta_3 = -2.0$: the negative sign is correctly identified, confirming that the model has learned that proximity to the station is valuable
- $\hat{\beta}_0 = 21.47$ vs true $\beta_0 = 30.0$: the intercept is less well recovered than the slopes, which is typical. The intercept represents the predicted price when all features are zero simultaneously, a combination that never appears in the data, so it is estimated with more uncertainty

The small discrepancies between estimated and true values are entirely expected: they reflect the noise ($\varepsilon$, standard deviation 20 £k) that was added during data generation. With more data, the estimates would converge closer to the true values.

**Evaluation metrics**

- **Test RMSE = 20.1 £k**: on average the model's predictions are off by roughly £20k on unseen houses. This is close to the noise standard deviation of 20 £k that was built into the data, which makes sense, since even the perfect model cannot do better than the irreducible noise.
- **Test $R^2$ = 0.908**: the model explains about 91% of the variation in house prices in the test set. The remaining 9% is attributable to the noise that no linear model can capture. An $R^2$ above 0.9 is generally considered a strong fit.

**Coefficient interpretation**

The plain-English interpretation at the bottom translates the coefficients into practical statements, always with the "holding all other features constant" caveat implied:

- A house 1 m² larger is predicted to sell for £1.55k more, assuming the same number of bedrooms and the same distance to the station
- A house with one extra bedroom is predicted to sell for £12k more, assuming the same floor area and distance
- A house 1 km further from the station is predicted to sell for £1.94k *less*; location matters, and the model has learned this correctly from the data

<br>

Since we cannot visualise a three-dimensional regression surface directly, the figure below shows the **marginal effect** of each feature individually. Each panel varies one feature across its full range while holding the other two fixed at their mean values, a standard technique for interpreting multiple regression models visually. The black line in each panel is the model's predicted price as that feature changes; the coloured dots are the actual test set observations.

Notice how the slope of the black line in each panel corresponds directly to the fitted coefficient: the area panel slopes gently upward (β₁ ≈ 1.55), the bedrooms panel has a steeper upward slope (β₂ ≈ 12.0), and the distance panel slopes downward (β₃ ≈ −1.94). The scatter of points around each line reflects the variation explained by the other two features: a test point can sit far from the line in one panel while still being well predicted overall, because the model uses all three features simultaneously.


In [ ]:
# Listing 15.
# ── Figure 7: Multiple regression — visualising the fit ───────────────────────
# With three features we cannot show the full fit in a single 2D plot, so we
# use three panels — one per feature — each showing that feature vs price with
# the model's predictions overlaid. The predictions vary across each panel
# because the other two features are held at their mean values (a standard
# way to visualise the marginal effect of one feature at a time).

fig, axes = plt.subplots(1, 3, figsize=(10, 5))
fig.canvas.header_visible = False
fig.canvas.resizable = True
fig.canvas.toolbar_visible = True
fig.canvas.toolbar_position = 'bottom'

feature_labels = {
    'area':     ('Floor area (m²)',         'steelblue'),
    'bedrooms': ('Number of bedrooms',      'seagreen'),
    'distance': ('Distance to station (km)', 'tomato'),
}

# Mean values of all three features — used to hold other features constant
# when plotting the marginal effect of each individual feature
mean_area     = df2['area'].mean()
mean_bedrooms = df2['bedrooms'].mean()
mean_distance = df2['distance'].mean()

for ax, (feat, (label, colour)) in zip(axes, feature_labels.items()):

    # Scatter: actual prices vs this feature for the test set
    feat_idx = feature_cols.index(feat)
    ax.scatter(X_te_mlr[:, feat_idx], y_te_mlr,
               color=colour, s=40, alpha=0.6, edgecolors='k', lw=0.3,
               zorder=3, label='Test data (actual price)')

    # Marginal effect line: vary this feature across its full range while
    # holding the other two features fixed at their mean values.
    feat_range = np.linspace(df2[feat].min(), df2[feat].max(), 200)

    # Build a (200, 3) feature matrix with this feature varying and the
    # others fixed at their means
    if feat == 'area':
        X_line = np.column_stack([feat_range,
                                   np.full(200, mean_bedrooms),
                                   np.full(200, mean_distance)])
    elif feat == 'bedrooms':
        X_line = np.column_stack([np.full(200, mean_area),
                                   feat_range,
                                   np.full(200, mean_distance)])
    else:
        X_line = np.column_stack([np.full(200, mean_area),
                                   np.full(200, mean_bedrooms),
                                   feat_range])

    y_line = mlr.predict(X_line)

    ax.plot(feat_range, y_line, color='black', linewidth=2.5, zorder=4,
            label=f'Model prediction\n(other features at mean)')

    ax.set_xlabel(label, fontsize=10)
    ax.set_ylabel('Sale price (£k)', fontsize=10)
    ax.set_title(
        f'Effect of {feat}\n'
        f'β = {mlr.coef_[feat_idx]:.2f} £k per unit',
        fontsize=10,
    )
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.2)

plt.suptitle(
    'Figure 7: Marginal effect of each feature on predicted price\n'
    '(other features held at their mean values)',
    fontsize=11,
)
plt.tight_layout(rect=[0, 0, 1, 0.93])
plt.show()

A good fit on the metrics alone is not sufficient: we also need to check that the model assumptions are not being violated. Figure 8 shows four residual diagnostic plots for the test set, each designed to reveal a different potential problem.

The **first panel** plots residuals against fitted values. If the linearity and homoscedasticity assumptions hold, this should look like random scatter centred on the red dashed zero line with no discernible pattern. A curve through the points would suggest the true relationship is non-linear; a funnel shape (where scatter increases with fitted value) would indicate heteroscedasticity, meaning the model's errors are larger for expensive houses than cheap ones.

The **remaining three panels** plot the residuals against each individual feature in turn. These are particularly useful for spotting features that have been modelled incorrectly. If the relationship between a feature and the target is genuinely linear, the residuals should show no pattern when plotted against that feature; they should scatter randomly above and below zero across the full range of values. If you see a U-shape or an S-shape, it suggests that feature has a non-linear effect that a simple linear term cannot capture, and a transformation (such as $x^2$ or $\log x$) might be needed.

For a dataset that was generated from a truly linear model, as ours was, all four panels should look like random noise around zero, confirming that the model assumptions are satisfied and the fit is appropriate.

In [ ]:
# Listing 16.
# ── Figure 8: MLR residual diagnostics ───────────────────────────────────────
# Residual plots reveal whether the model assumptions are satisfied and whether
# any feature has a non-linear relationship that the model has missed.
test_residuals_mlr = y_te_mlr - y_hat_te_mlr

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
fig.canvas.header_visible = False
fig.canvas.resizable = True
fig.canvas.toolbar_visible = True
fig.canvas.toolbar_position = 'bottom'

# ── Panel 1: residuals vs fitted values ──────────────────────────────────────
# Should show random scatter around zero with no pattern.
# A curve or funnel shape suggests the linear assumption is violated.
ax = axes[0]
ax.scatter(y_hat_te_mlr, test_residuals_mlr, color='steelblue', s=40,
           alpha=0.7, edgecolors='k', lw=0.3)
ax.axhline(0, color='red', linewidth=1.5, linestyle='--')
ax.set_xlabel(r'Fitted values $\hat{y}$')
ax.set_ylabel('Residuals (£k)')
ax.set_title('Residuals vs Fitted\n(no pattern expected)')
ax.grid(True, alpha=0.2)

# ── Panels 2–4: residuals vs each individual feature ─────────────────────────
# If a feature has been correctly modelled as linear, its residual plot should
# show random scatter. A curve suggests a non-linear term is needed.
feature_colours = ['steelblue', 'seagreen', 'tomato']
feature_labels  = ['Floor area (m²)', 'Bedrooms', 'Distance (km)']

for i, (ax, feat_idx, label, colour) in enumerate(
    zip(axes[1:], range(3), feature_labels, feature_colours)
):
    ax.scatter(X_te_mlr[:, feat_idx], test_residuals_mlr,
               color=colour, s=40, alpha=0.7, edgecolors='k', lw=0.3)
    ax.axhline(0, color='red', linewidth=1.5, linestyle='--')
    ax.set_xlabel(label)
    ax.set_ylabel('Residuals (£k)')
    ax.set_title(f'Residuals vs {label.split(" ")[0]}\n(no pattern expected)')
    ax.grid(True, alpha=0.2)

plt.suptitle(
    f'Figure 8: MLR residual diagnostics (test set)  |  '
    f'Test RMSE = {np.sqrt(mse_test):.1f} £k  |  Test R² = {r2_test:.3f}',
    fontsize=11,
)
plt.tight_layout(rect=[0, 0, 1, 0.93])
plt.show()


---

## 6. From Regression to Classification

🔙 [Back to Table of Contents](#Table-of-Contents)

### 6.1 Why Linear Regression Cannot Reliably Classify

Linear regression is well suited to predicting **continuous** outcomes: quantities like house prices, temperatures, or blood pressure readings that can take any value along a scale. But what if we want to *classify* a data point, predicting whether it belongs to class 0 or class 1?

At first glance, applying linear regression to a binary label seems reasonable. If we encode class 0 as $y = 0$ and class 1 as $y = 1$, we could fit a line through the data and use a threshold (say, $\hat{y} = 0.5$) to decide which class to predict. For a clean, well-separated dataset this can even work adequately. However, there are two fundamental problems that make linear regression an unreliable and theoretically inappropriate tool for classification:

**Problem 1 — Unbounded predictions**

A straight line extends infinitely in both directions. For data points at the extreme ends of the feature range, linear regression will routinely produce predicted values well outside the interval $[0, 1]$, for example $\hat{y} = -0.3$ or $\hat{y} = 1.7$. These values have no meaningful interpretation as class membership probabilities. A probability of $-0.3$ is mathematically undefined, and a threshold-based classifier built on such predictions is making decisions based on numbers that have no probabilistic meaning.

**Problem 2 — Instability to new data**

Linear regression fits a line that minimises the sum of squared residuals across all training points simultaneously. This means the position of the decision boundary depends on every point in the dataset, including points that are far from the boundary and clearly deep inside one class. Adding a single new extreme data point (a very large or very small feature value) forces a complete refit of the entire line, which can shift the decision boundary dramatically even though the underlying classification problem has not changed. A robust classifier should be largely unaffected by points that are already confidently classified.

<br>

Both of these problems stem from the same root cause: linear regression was designed to model the **expected value** of a continuous outcome, not the **probability** of a binary event. What we need instead is a model that is constrained to produce outputs between 0 and 1, is stable in the presence of extreme data points, and has a natural probabilistic interpretation. Thankfully, such a model exists. That model is **logistic regression**, which I'll introduce in the next section.

Figure 8 below demonstrates both problems I discuss above visually. The left panel (Figure 8a) shows linear regression fitted directly to binary labels: note the orange shaded regions where the predicted values fall outside $[0, 1]$, making them impossible to interpret as probabilities. The right panel (Figure 8b) is interactive: drag the slider to move a single extreme data point further along the x-axis and watch how the refitted decision boundary shifts in response. The original boundary position is shown as a faint solid line for comparison. The further the extreme point, the larger the shift, even though the other 80 data points have not changed at all.

In [ ]:
# Listing 17.
%matplotlib widget
from visualisations.Figure_8 import show
show()

### 6.2 The Sigmoid Function

The two problems with linear regression identified in the previous section both stem from the same root cause: the linear model produces outputs on an unbounded scale, but classification requires outputs that can be interpreted as probabilities, numbers constrained to lie between 0 and 1. The solution is to wrap the linear output in a function that performs exactly this mapping: no matter what we pass into it, it will return a value between 0 and 1.

We need a function that takes any number (positive, negative, large, small) and squashes it into a value between 0 and 1 that can be interpreted as a **probability**. There is one such function that does this. It's called the **sigmoid function**:

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

Before explaining what this does, it helps to understand the $e$ in the formula. $e$ is a mathematical constant approximately equal to 2.718, and it appears throughout mathematics and nature in the same way that $\pi$ appears in geometry. The key property we use here is that $e^{-z}$ (read "e to the power of minus z") gets very small when $z$ is large and positive, and very large when $z$ is large and negative.

With that in mind, let's consider what the sigmoid does at a few specific values of $z$:

- When $z = 0$: $e^{-0} = 1$, so $\sigma(0) = \frac{1}{1 + 1} = 0.5$, the midpoint, representing maximum uncertainty
- When $z$ is large and positive (say $z = 10$): $e^{-10} \approx 0$, so $\sigma(10) = \frac{1}{1 + 0} \approx 1$, meaning the model is highly confident this is class 1
- When $z$ is large and negative (say $z = -10$): $e^{10}$ is enormous, so $\sigma(-10) = \frac{1}{1 + \text{large number}} \approx 0$, meaning the model is highly confident this is class 0

In other words, the sigmoid takes any number on the number line and maps it to a value between 0 and 1, creating an S-shaped curve. No matter how extreme the input, the output can never reach exactly 0 or exactly 1; it just gets arbitrarily close. This is exactly what we need: a model output that behaves like a probability.

The figure below lets you explore the sigmoid function interactively before we use it inside a full logistic regression model.

**Left panel** — the sigmoid curve plotted across a range of $z$ values from −8 to +8. The blue shaded region shows where the model would predict class 0 ($\sigma(z) \leq 0.5$); the red shaded region shows where it would predict class 1 ($\sigma(z) > 0.5$). The red dashed line at 0.5 is the decision threshold: the point of maximum uncertainty where the model is equally torn between the two classes.

**Right panel** — shows the sigmoid calculation with your current $z$ value substituted in step by step, a visual bar indicating where the predicted probability sits between 0 and 1, and a plain-English summary of the prediction and how confident the model is.

**Drag the slider** to move $z$ left and right and observe the following:

- Near $z = 0$ the dot sits at the midpoint of the S-curve and the probability is close to 0.5, meaning the model is uncertain
- As $z$ increases positively the probability climbs steeply toward 1, meaning the model becomes increasingly confident the observation is class 1
- As $z$ decreases below zero the probability falls toward 0, meaning the model becomes increasingly confident the observation is class 0
- Beyond roughly $z = \pm 4$ the curve flattens out, so additional increases in $z$ produce very little change in probability because the model is already close to certain. This flat region is called **saturation** and has important consequences for how logistic regression is trained.

Notice that the output is always strictly between 0 and 1. No matter how large or small $z$ becomes, the sigmoid never actually reaches 0 or 1. This is precisely the property that linear regression lacked.

In [ ]:
# Listing 18.
%matplotlib widget
from visualisations.Figure_9 import show
show()

In logistic regression, $z$ is not a fixed number: it is computed from the input features of a single example, using the same linear combination as multiple regression:

$$z = \beta_0 + \beta_1 x_1 + \cdots + \beta_p x_p$$

This is the part of the model that is *fitted* during training. But what does fitting mean here? Recall that in linear regression, we found $\beta_0$ and $\beta_1$ by minimising the sum of squared residuals, the gap between predicted and actual values. In logistic regression the target is a class label (0 or 1), so we cannot measure error in the same way. Instead, we ask a different question: **given the training data, where we know the true class label for every observation, what values of $\beta_0, \beta_1, \ldots, \beta_p$ make the model assign high probability to the correct class for as many training examples as possible?**

For example, if a training observation has a true label of 1, we want $\sigma(z)$ to be close to 1 for that observation. If the true label is 0, we want $\sigma(z)$ to be close to 0. Training finds the parameter values that collectively achieve this across the entire training set. This approach is called **maximum likelihood estimation** and is the standard method for fitting logistic regression. Scikit-learn handles this automatically.

Passing the linear combination through the sigmoid gives the predicted **probability** that the observation belongs to class 1:

$$P(Y = 1 \mid x) = \sigma(z) = \frac{1}{1 + e^{-(\beta_0 + \beta_1 x_1 + \cdots + \beta_p x_p)}}$$

The classification rule is then straightforward: if $\sigma(z) > 0.5$, the model predicts class 1; if $\sigma(z) \leq 0.5$, it predicts class 0. This threshold of 0.5 corresponds exactly to $z = 0$, the point where the model is equally uncertain between the two classes. It is the default threshold but can be adjusted.

### 6.3 The Logit Transformation

The sigmoid solves the problem of unbounded predictions, allowing us to assign probabilities, but it creates a new problem too. Once we pass $z$ through the sigmoid, the relationship between the raw feature $x$ and the predicted probability is no longer a straight line. We get the S-shaped curve, not a line. A linear model can only produce straight lines, so we can't learn this curve directly. So how can we use a linear model to learn a shape that is fundamentally curved?

The answer lies in a clever mathematical trick. Instead of modelling the probability $P(Y=1)$ directly, we model a *transformation* of it that happens to be linear in the feature space. This transformation is called the **logit function**, the inverse of the sigmoid.

#### Understanding odds first

Before introducing the logit, it helps to understand **odds**, a way of expressing probability that you may have encountered in betting. If the probability of an event is $P$, the odds of that event are:

$$\text{odds} = \frac{P}{1 - P}$$

If $P = 0.75$ (a 75% chance of class 1), the odds are $\frac{0.75}{0.25} = 3$, meaning class 1 is three times more likely than class 0. If $P = 0.5$, the odds are $\frac{0.5}{0.5} = 1$, meaning both classes are equally likely. Odds can range from 0 (impossible) to infinity (certain), but cannot be negative.

Taking the natural logarithm of the odds gives the **log-odds**, also called the **logit**:

$$z = \log\left(\frac{P(Y=1)}{1 - P(Y=1)}\right)$$

The logarithm stretches the odds, which range from 0 to infinity, back out onto the entire real number line from $-\infty$ to $+\infty$. This is important: while probabilities are constrained to $[0, 1]$ and odds are constrained to $[0, \infty)$, log-odds are unconstrained, meaning they can be any real number. And unconstrained real numbers are exactly what a linear model produces.

#### Why does this help?

Because in log-odds space, the relationship between $z$ and the features is linear:

$$\log\left(\frac{P(Y=1)}{1 - P(Y=1)}\right) = \beta_0 + \beta_1 x_1 + \cdots + \beta_p x_p$$

This is the key insight of logistic regression: we are **not** trying to fit a straight line directly to the probabilities (which would produce the problems we saw with linear regression). Instead, we are fitting a straight line to the *log-odds* of the probability, a quantity that genuinely is linear in the features. When we need to make a prediction, we take the linear output, pass it through the sigmoid to convert log-odds back into a probability, and then apply the 0.5 threshold to get a class label.

To summarise the full pipeline:

1. Compute $z = \beta_0 + \beta_1 x_1 + \cdots + \beta_p x_p$, a linear combination of the features (this is the same as multiple regression)
2. Pass $z$ through the sigmoid: $P(Y=1) = \sigma(z) = \frac{1}{1 + e^{-z}}$, which converts log-odds to probability
3. Apply the threshold: predict class 1 if $P(Y=1) > 0.5$, otherwise predict class 0

Training finds the values of $\beta_0, \beta_1, \ldots, \beta_p$ such that step 2 assigns high probability to the correct class for as many training examples as possible. The linearity happens in step 1, in log-odds space; the curve you see in the original feature space is the result of the sigmoid transformation in step 2.

---

The figure below shows the three-step pipeline of logistic regression (the linear model, the sigmoid transformation, and the logit inverse), all linked together so you can see how a change in one panel ripples through the others.

**Left panel — the linear model in log-odds space.** This is the straight line $z = \beta_0 + \beta_1 x$, the part of the model that is actually fitted during training. It is unbounded and can take any value, which is why a linear model is appropriate here.

**Middle panel — after the sigmoid transformation.** The same straight line from the left panel, passed through $\sigma(z) = \frac{1}{1 + e^{-z}}$. The straight line becomes the familiar S-curve. The blue shaded region shows where the model predicts class 0 and the red region where it predicts class 1, with the dashed red line at $P = 0.5$ as the decision boundary.

**Right panel — the logit (inverse sigmoid).** This panel shows the logit function going in the opposite direction: starting with a probability and converting it back to a log-odds value. It is the mirror image of the sigmoid, confirming that the two functions are true inverses of each other. The printed readout below the figure verifies this numerically for your tracked point.

**Use the three controls:**

- **Slope (β₁)**: increasing the slope makes the sigmoid curve steeper, meaning the model transitions between class 0 and class 1 more sharply. A very flat slope produces a gradual, uncertain transition; a steep slope produces a sharp, confident one. Notice how the straight line in the left panel tilts and the S-curve in the middle panel changes shape simultaneously.
- **Intercept (β₀)**: shifts the line left or right in the left panel, which moves the S-curve and its decision boundary left or right in the middle panel. This is equivalent to changing where on the x-axis the model is 50% confident.
- **Track x**: moves a red dot across all three panels simultaneously for a specific feature value. The printed readout below shows the complete calculation: the linear combination gives a log-odds value, the sigmoid converts it to a probability, and the logit converts it back, confirming the round-trip and showing every intermediate step with real numbers.

In [ ]:
# Listing 19.
%matplotlib widget
from visualisations.Figure_10 import show
show()

Let's step back and think through this as if you'd just collected a dataset and wanted to build a classifier yourself. You have observations (houses, patients, fruit) each described by one or more features and each belonging to one of two classes. What do you actually do?

**You start with the data.** Each observation gives you a vector of feature values $x_1, x_2, \ldots, x_p$ and a true class label $y$ (0 or 1). During training, you know both: you have the features and the correct answer. Your goal is to find a model that learns the relationship between them so it can predict the correct label for new observations where you only have the features.

**Step 1 — what are you actually fitting, and where?** This is the part that confuses most people. You are not fitting a line directly to the class labels (0 or 1). Instead, you are fitting a line in **log-odds space**. But how do you get to log-odds space? You do not start there; the training algorithm works backwards from the true class labels. For each training observation, the true label $y$ is either 0 or 1, and we treat these as probabilities: $y=1$ means $P(Y=1) = 1$ and $y=0$ means $P(Y=1) = 0$. Applying the logit to these probabilities would give $+\infty$ and $-\infty$ respectively, which is impractical to work with directly. So instead, the training algorithm asks a softer question: what values of $\beta_0$ and $\beta_1$ make the sigmoid output as close as possible to the true labels, across all training observations simultaneously? The parameters $\beta_0$ and $\beta_1$ define the straight line in the left panel. They are what you are searching for, and they live in log-odds space, because the sigmoid maps log-odds to probabilities.

**Step 2 — convert $z$ into something comparable to the label.** Once you have a candidate set of parameters $\beta_0$ and $\beta_1$, you compute $z = \beta_0 + \beta_1 x$ for each training observation and pass it through the sigmoid (middle panel) to get $P(Y=1)$, a number between 0 and 1 representing the model's confidence that the observation belongs to class 1. Now you have something you can compare to the true label: if the true label is 1 and the model says $P(Y=1) = 0.9$, that is a good prediction; if it says $P(Y=1) = 0.1$, that is a bad prediction. This comparison is what gives training its feedback signal.

**Step 3 — measure the error and update the parameters.** With a predicted probability and a known true label for every training observation, the model computes a loss called **log-loss** (also called cross-entropy), which penalises confident wrong predictions very heavily. The gradient of this loss with respect to $\beta_0$ and $\beta_1$ tells the optimiser which direction to adjust the parameters to reduce the error, exactly the gradient descent process from Notebook 4, but now applied through the sigmoid. This loop of predicting, measuring error, and updating parameters repeats until the loss stops improving. At that point training is complete and $\beta_0$ and $\beta_1$ are fixed.

**Step 4 — making a prediction on new data.** At inference time you only have features, no label. You compute $z$ from the fitted $\beta_0$ and $\beta_1$ (left panel), pass it through the sigmoid (middle panel), and if the resulting probability exceeds 0.5 you predict class 1, otherwise class 0.

**So what is the right panel for?** The logit (right panel) is not a step you take at prediction time. It is there to show you *why fitting a line in log-odds space is the right thing to do*. The logit is the inverse of the sigmoid: it takes a probability and maps it back to log-odds. The fact that this mapping produces a straight line, exactly the line you fitted in the left panel, confirms that the linear model in log-odds space is not an arbitrary choice or an approximation. It is the mathematically exact consequence of wanting the model's output to behave like a well-calibrated probability. The right panel is the justification; the left and middle panels are the practice.

### 6.4 Logistic Regression with scikit-learn

Putting the sigmoid and the logit together, the full logistic regression pipeline is:

1. Choose initial values for $\beta_0, \beta_1, \ldots, \beta_p$.
2. For each training sample $x_i$, compute the linear combination: $z_i = \beta_0 + \beta_1 x_{i1} + \cdots + \beta_p x_{ip}$.
3. Pass $z_i$ through the sigmoid to obtain a predicted probability: $\hat{p}_i = \sigma(z_i) = P(Y=1 \mid x_i)$.
4. Apply the classification rule: predict class 1 if $\hat{p}_i > 0.5$, else class 0.
5. Compare the prediction to the true label $y_i$ and update the coefficients if there is an error.
6. Repeat until the objective (the loss function, discussed in Section 6.6) is minimised.

---

#### Training the Model

Training logistic regression in scikit-learn follows the same three-step pattern used throughout these notebooks:

```python
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

# 1. Split the data into training and test sets
#    test_size=0.2 reserves 20% of the data for evaluation
#    random_state fixes the random seed for reproducibility
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 2. Create the model
#    solver='lbfgs' is a good default optimiser for small-to-medium datasets
#    max_iter controls how many optimisation steps are allowed before stopping
model = LogisticRegression(solver='lbfgs', max_iter=1000)

# 3. Fit the model: this is where the learning happens
#    scikit-learn uses Maximum Likelihood Estimation (MLE) internally,
#    finding the coefficients that maximise the probability of observing
#    the training labels given the features. The optimisation is handled
#    automatically, with no manual gradient descent required.
model.fit(X_train, y_train)
```

Note that scikit-learn's `LogisticRegression` fits the parameters using **Maximum Likelihood Estimation (MLE)** rather than the sum of squared residuals used in linear regression. MLE finds the $\beta_0$ and $\beta_1$ that maximise the probability of observing the training labels given the features, which is equivalent to minimising the log-loss. An iterative optimisation algorithm handles this automatically.

---

#### What the Model Has Learned

After calling `fit()`, the trained model exposes its learned parameters as attributes you can inspect directly:

```python
model.coef_       # the learned coefficients β_1, ..., β_p, one per feature
model.intercept_  # the learned intercept β_0
model.classes_    # the class labels the model was trained on
```

- `coef_` contains the learned coefficients, one per input feature. A large positive coefficient means that feature strongly pushes the predicted probability toward class 1; a large negative coefficient pushes it toward class 0. A coefficient near zero means that feature had little influence on the boundary.
- `intercept_` is the learned bias term $\beta_0$. It shifts the decision boundary without changing its orientation, exactly as $b$ does in the SVM.

---

#### Making Predictions and Getting Probabilities

```python
# Predict the class label for each test example
#   Returns 0 or 1 for each row, whichever class has predicted probability > 0.5
y_pred = model.predict(X_test)

# Predict the probability of each class for each test example
#   Returns a matrix of shape (n_samples, n_classes)
#   Each row sums to 1; column 0 is P(benign), column 1 is P(malignant)
y_proba = model.predict_proba(X_test)
```

`predict_proba()` is one of the most useful outputs of logistic regression. Unlike a decision tree which returns the fraction of training examples of each class at a leaf, logistic regression returns a smooth, calibrated probability, meaning a predicted probability of 0.85 genuinely reflects higher confidence than one of 0.55. This makes logistic regression particularly well suited to tasks where the probability itself matters, not just the class label.

---

#### Evaluating the Model

```python
from sklearn.metrics import accuracy_score, classification_report

# Overall fraction of test examples classified correctly
accuracy = accuracy_score(y_test, y_pred)
print(f'Test accuracy: {accuracy:.1%}')

# Detailed per-class breakdown of precision, recall, and F1 score
#   (these metrics are covered in depth in Notebooks 10 and 11)
print(classification_report(y_test, y_pred))
```

---

#### The Dataset: Tumour Classification

The example below applies this pipeline to a **tumour classification** problem: given a tumour's size in mm, predict whether it is malignant (1) or benign (0).

The dataset is synthetic but based on a realistic scenario. Two groups of tumours are generated, each following a normal distribution:

| Variable | Description | Type | Values |
|---|---|---|---|
| `x_tumour` | Tumour diameter in mm, the single input feature | Continuous | Benign: mean 15 mm, std 4 mm; Malignant: mean 25 mm, std 4 mm |
| `y_tumour` | True class label: whether the tumour is malignant | Binary | 0 = benign, 1 = malignant |
| `n` | Total number of observations | Integer | 200 (100 benign, 100 malignant) |

The two groups overlap: a benign tumour can be larger than some malignant ones, and vice versa. This is deliberately realistic, since no single threshold perfectly separates the two classes, and some misclassification is unavoidable. The task of logistic regression is to find the decision boundary in tumour size that minimises these errors on average.

---

#### Key Hyperparameters

| Parameter | Default | What it controls |
|---|---|---|
| `C` | `1.0` | Inverse of regularisation strength: smaller values apply stronger regularisation, shrinking the coefficients toward zero and reducing overfitting |
| `solver` | `'lbfgs'` | The optimisation algorithm used internally. `'lbfgs'` is a good default; `'saga'` is faster on very large datasets |
| `max_iter` | `100` | Maximum number of optimisation iterations. Increase if the model reports a convergence warning |
| `penalty` | `'l2'` | The type of regularisation applied: `'l2'` penalises large coefficients; `'l1'` can drive some coefficients to exactly zero (feature selection) |
| `class_weight` | `None` | Set to `'balanced'` to compensate for imbalanced classes |
| `multi_class` | `'auto'` | How to handle more than two classes: `'ovr'` trains one classifier per class; `'multinomial'` fits a single model across all classes simultaneously |

The most important parameter to tune is `C`. A simple manual sweep is a reasonable starting point:

```python
for c in [0.01, 0.1, 1.0, 10.0, 100.0]:
    m = LogisticRegression(C=c, solver='lbfgs', max_iter=1000)
    m.fit(X_train, y_train)
    acc = accuracy_score(y_test, m.predict(X_test))
    print(f'C={c:<8}  →  accuracy={acc:.4f}')
```

#### Regularisation in Logistic Regression

Regularisation is a technique for preventing overfitting by penalising the model for having large coefficient values. The idea is simple: if a coefficient grows very large, the model has become highly sensitive to that particular feature, so small changes in its value produce large swings in the predicted probability. This is rarely desirable. A model that is too finely tuned to the training data will generalise poorly to new examples.

In logistic regression, regularisation is controlled by the parameter `C`, the *inverse* of the regularisation strength. This is slightly counterintuitive at first:

- **Large C**: weak regularisation. The model is free to make coefficients as large as it needs to fit the training data well. Higher risk of overfitting.
- **Small C**: strong regularisation. The model is penalised heavily for large coefficients, which forces them toward zero. The decision boundary becomes simpler and more conservative. Lower risk of overfitting, but the model may underfit if `C` is too small.

The default value of `C=1.0` is a reasonable middle ground for most problems, but it is worth tuning on a held-out validation set.

---

#### L1 vs L2 Regularisation

Scikit-learn supports two types of regularisation, controlled by the `penalty` parameter:

**L2 regularisation** (`penalty='l2'`, the default) adds a penalty proportional to the *square* of each coefficient:

$$\text{penalty} = \frac{1}{2C} \sum_{j=1}^{p} \beta_j^2$$

L2 pushes all coefficients toward zero but rarely drives any of them exactly to zero. Every feature retains some influence, but large coefficients are suppressed. This is the most common choice and works well in most situations.

**L1 regularisation** (`penalty='l1'`) adds a penalty proportional to the *absolute value* of each coefficient:

$$\text{penalty} = \frac{1}{C} \sum_{j=1}^{p} |\beta_j|$$

L1 has a qualitatively different effect: it tends to drive some coefficients to exactly zero, effectively removing those features from the model entirely. This makes L1 useful as an automatic feature selection tool. If you have many features and suspect only a few are truly relevant, L1 regularisation can identify them by zeroing out the rest.

The practical difference can be illustrated simply:

```python
from sklearn.linear_model import LogisticRegression

# L2: shrinks all coefficients but keeps all features
model_l2 = LogisticRegression(penalty='l2', C=0.1, solver='lbfgs')
model_l2.fit(X_train, y_train)
print('L2 coefficients:', model_l2.coef_)

# L1: drives some coefficients to exactly zero (requires saga solver)
model_l1 = LogisticRegression(penalty='l1', C=0.1, solver='saga', max_iter=1000)
model_l1.fit(X_train, y_train)
print('L1 coefficients:', model_l1.coef_)
# Any coefficient that is exactly 0.0 means that feature was discarded
```

In practice, for a dataset with a small number of well-chosen features, like the tumour size example below, the choice between L1 and L2 rarely makes a meaningful difference. The distinction matters most when you have many features, some of which may be redundant or irrelevant. In that setting, L1 regularisation can be a valuable first step before applying more complex models.

In [ ]:
# Listing 20.
# ── Step 1: Generate the synthetic tumour dataset ─────────────────────────────
rng4 = np.random.default_rng(11)
n4   = 200   # 200 tumours in total — 100 benign, 100 malignant

# Benign tumours tend to be smaller — centred around 15 mm with std of 4 mm.
# Malignant tumours tend to be larger — centred around 25 mm with std of 4 mm.
# The distributions overlap, so some tumours will be misclassified — this
# is realistic and intentional.
benign    = rng4.normal(15, 4, n4 // 2)   # 100 benign tumour sizes (mm)
malignant = rng4.normal(25, 4, n4 // 2)   # 100 malignant tumour sizes (mm)

# Combine both groups into a single feature array.
# .reshape(-1, 1) converts the 1D array of 200 values into a 2D column vector
# of shape (200, 1) — scikit-learn expects a 2D feature matrix.
x_tumour = np.concatenate([benign, malignant]).reshape(-1, 1)

# Create the corresponding labels: 0 = benign, 1 = malignant.
# The order matches x_tumour — first 100 are benign (0), last 100 are malignant (1).
y_tumour = np.array([0] * (n4 // 2) + [1] * (n4 // 2))

# ── Step 2: Train/test split ──────────────────────────────────────────────────
# Reserve 20% (40 tumours) for evaluation. random_state=0 ensures the same
# split every time the cell is run.
X_tr4, X_te4, y_tr4, y_te4 = train_test_split(
    x_tumour, y_tumour, test_size=0.2, random_state=0
)

# ── Step 3: Fit the logistic regression model ─────────────────────────────────
# LogisticRegression() fits β₀ and β₁ using Maximum Likelihood Estimation —
# it finds the parameters that make the sigmoid output as close as possible
# to the true labels across all training observations.
# The interface is identical to LinearRegression: create, fit, predict.
log_reg = LogisticRegression()
log_reg.fit(X_tr4, y_tr4)

# After fitting, the learned parameters are stored as:
#   log_reg.intercept_[0]   — β₀ (note: stored as a 1-element array, hence [0])
#   log_reg.coef_[0][0]     — β₁ (stored as a 2D array, hence [0][0])
print('Logistic Regression — trained coefficients')
print(f'  β₀ (intercept)      = {log_reg.intercept_[0]:.4f}')
print(f'  β₁ (size coeff.)    = {log_reg.coef_[0][0]:.4f}')
print()

# ── Step 4: Make predictions manually using the sigmoid ───────────────────────
# Rather than calling log_reg.predict() directly, we compute predictions
# step by step to reinforce the pipeline: z → sigmoid → threshold.

# Define the sigmoid function — maps any real number z to a probability in (0, 1)
def sigmoid(z):
    return 1 / (1 + np.exp(-z))
    
# Step 4a: compute z = β₀ + β₁ × tumour_size for every test observation.
# X_te4.ravel() converts the (40,1) column vector back to a flat 1D array
# so the arithmetic works element-wise.
z_test = log_reg.intercept_[0] + log_reg.coef_[0][0] * X_te4.ravel()

# Step 4b: pass z through the sigmoid to get the predicted probability of
# malignancy — P(Y=1 | tumour size) — for each test observation.
p_test = sigmoid(z_test)

# Step 4c: apply the decision threshold.
# (p_test >= 0.5) produces a boolean array (True/False).
# .astype(int) converts True → 1 (predict malignant) and False → 0 (predict benign).
y_pred4 = (p_test >= 0.5).astype(int)

# ── Step 5: Evaluate on the test set ─────────────────────────────────────────
# accuracy_score counts the fraction of correct predictions.
print('Test set performance')
print(f'  Accuracy   = {accuracy_score(y_te4, y_pred4):.1%}')

# The confusion matrix breaks down predictions into four categories:
#   TN (True Negative):  benign tumour correctly predicted as benign
#   FP (False Positive): benign tumour incorrectly predicted as malignant
#   FN (False Negative): malignant tumour incorrectly predicted as benign ← most dangerous
#   TP (True Positive):  malignant tumour correctly predicted as malignant
# In a medical context, False Negatives are particularly costly — missing
# a malignant tumour has far worse consequences than a false alarm.
cm = confusion_matrix(y_te4, y_pred4)
print(f'  Confusion matrix:')
print(f'    TN = {cm[0, 0]}   FP = {cm[0, 1]}')
print(f'    FN = {cm[1, 0]}   TP = {cm[1, 1]}')

---

The plots below visualise the two key outputs of logistic regression.

The **left panel** shows the fitted sigmoid curve overlaid on the raw data: blue dots are benign tumours and red dots are malignant ones, both plotted at $y = 0$ or $y = 1$ respectively. The S-shaped curve shows the model's predicted probability of malignancy at every possible tumour size. Notice how the curve smoothly transitions from near-zero (confidently benign) to near-one (confidently malignant) as tumour size increases. The vertical dashed line marks the **decision boundary**: the tumour size at which $P(\text{malignant}) = 0.5$, the point where the model switches its prediction from benign to malignant.

The **right panel** shows the **log-odds line**, the actual straight line that logistic regression fits to the data. This is the left panel from Figure 10: a linear relationship in log-odds space. Applying the sigmoid to every point on this line produces the S-shaped curve in the left panel. The two panels together show both sides of the same model.

**Understanding the confusion matrix** printed in the output above. When a classifier makes predictions on the test set, each prediction falls into one of four categories:

| | Predicted benign (0) | Predicted malignant (1) |
|---|---|---|
| **Actually benign (0)** | True Negative (TN) ✓ | False Positive (FP) ✗ |
| **Actually malignant (1)** | False Negative (FN) ✗ | True Positive (TP) ✓ |

- **True Negatives (TN)**: benign tumours correctly identified as benign
- **True Positives (TP)**: malignant tumours correctly identified as malignant
- **False Positives (FP)**: benign tumours incorrectly flagged as malignant (a false alarm)
- **False Negatives (FN)**: malignant tumours incorrectly classified as benign (a missed diagnosis)

In a medical context, False Negatives are by far the most dangerous category: a missed malignant tumour could mean a patient does not receive timely treatment. This is one of the reasons that accuracy alone is often an insufficient metric for classification problems, and why we will return to precision, recall, and other metrics in later notebooks.


In [ ]:
# Listing 21.
# ── Compute the fitted sigmoid curve and log-odds line ────────────────────────
# x_range covers the full range of tumour sizes for smooth curve drawing.
# reshape(-1, 1) converts to a column vector as required by scikit-learn.
x_range = np.linspace(0, 45, 400).reshape(-1, 1)

# Compute z = β₀ + β₁x for every point in x_range
z_range = log_reg.intercept_[0] + log_reg.coef_[0][0] * x_range.ravel()

# Pass z through the sigmoid to get predicted probabilities
p_range = sigmoid(z_range)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
fig.canvas.toolbar_visible = True
fig.canvas.toolbar_position = 'bottom'
fig.canvas.header_visible = False

# ── Left panel: sigmoid curve overlaid on the raw data ───────────────────────
ax = axes[0]

# Plot each group at y=0 (benign) and y=1 (malignant) — all on the same x-axis
# so the sigmoid curve can be overlaid directly on the class labels
ax.scatter(benign,    np.zeros(n4 // 2), color='steelblue', s=25,
           alpha=0.5, edgecolors='k', lw=0.3, label='Benign (0)', zorder=3)
ax.scatter(malignant, np.ones(n4 // 2),  color='tomato',    s=25,
           alpha=0.5, edgecolors='k', lw=0.3, label='Malignant (1)', zorder=3)

# The fitted sigmoid curve — P(malignant | tumour size) at every x value.
# Raw strings (r'...') prevent Python from interpreting backslashes as
# escape sequences, which avoids the SyntaxWarning on LaTeX labels.
ax.plot(x_range, p_range, color='seagreen', linewidth=2.5,
        label=r'$P(\mathrm{malignant} \mid x)$')

ax.axhline(0.5, color='black', linewidth=1.5, linestyle='--',
           label='Threshold = 0.5')

# Decision boundary: the tumour size where P = 0.5, i.e. where z = 0.
# Setting z = 0: β₀ + β₁x = 0 → x = -β₀ / β₁
x_boundary = -log_reg.intercept_[0] / log_reg.coef_[0][0]
ax.axvline(x_boundary, color='red', linewidth=1.5, linestyle=':',
           label=f'Decision boundary: {x_boundary:.1f} mm')

ax.set_xlabel('Tumour size (mm)')
ax.set_ylabel(r'$P(Y=1 \mid x)$ — probability of malignancy')
ax.set_title('Figure 11a: Logistic regression — fitted sigmoid')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.2)

# ── Right panel: the log-odds line (what is actually fitted) ──────────────────
# This is the linear part of the model — a straight line in log-odds space.
# Applying the sigmoid to this line produces the S-curve in the left panel.
ax = axes[1]

ax.plot(x_range, z_range, color='seagreen', linewidth=2.5,
        label=rf'$z = {log_reg.intercept_[0]:.2f} + {log_reg.coef_[0][0]:.3f}x$')

# The red dashed line at z=0 corresponds to P=0.5 — the decision boundary.
# Every point above this line is predicted malignant; below is predicted benign.
ax.axhline(0, color='red', linewidth=1.5, linestyle='--',
           label=r'$z = 0 \Rightarrow \sigma(z) = 0.5$ (boundary)')

ax.set_xlabel('Tumour size (mm)')
ax.set_ylabel(r'$z$ = log-odds')
ax.set_title('Figure 11b: The log-odds line (what is actually fitted)\n'
             'Apply sigmoid to get probability')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.show()

---

### 6.5 Why Least Squares Fails for Logistic Regression

In section 4.4 we minimised the sum of squared residuals to fit a line. Why can we not do the same here?

Recall that logistic regression works in log-odds space, and log-odds become infinite as probabilities approach 0 or 1. A benign tumour with a very small malignancy probability might have a log-odds of −10, −100, or even −1000. If we tried to fit a line by minimising squared residuals in this space, these extreme values would produce enormous squared errors that would completely dominate the loss function, making the coefficient estimates unstable and meaningless. The squared error loss was designed for bounded, continuous outputs, and it simply breaks down when the target values can be infinite.

Logistic regression therefore uses a fundamentally different objective: **Maximum Likelihood Estimation (MLE)**.

### 6.6 Maximum Likelihood Estimation

The idea behind MLE is intuitive: find the parameter values that make the observed training data look as *probable* as possible under the model. In other words, if the model had those parameters, what is the chance it would have produced exactly the training labels we observed? The best parameters are the ones that maximise this chance.

**Individual observation likelihood.** For a single training observation, the model outputs a predicted probability $\hat{p}_i = P(Y=1 \mid x_i)$, the model's confidence that observation $i$ is class 1. How do we judge whether this prediction is good?

- If the true label is $y_i = 1$ (malignant): a good model should give a high $\hat{p}_i$. The likelihood of seeing this label under the model is simply $\hat{p}_i$: the higher the predicted probability for the correct class, the better.
- If the true label is $y_i = 0$ (benign): a good model should give a low $\hat{p}_i$. The likelihood of seeing this label is $1 - \hat{p}_i$: the lower the predicted probability of malignancy, the better.

For example: if a tumour is truly malignant ($y=1$) and the model predicts $\hat{p} = 0.9$, the likelihood is 0.9, which is good. If the model predicts $\hat{p} = 0.2$ for the same malignant tumour, the likelihood is only 0.2, which is poor.

These two cases can be written as one compact expression using a mathematical trick. The **total likelihood** across all $n$ training observations is the product of all individual likelihoods:

$$L(\boldsymbol{\beta}) = \prod_{i=1}^{n} \hat{p}_i^{\,y_i} \left(1 - \hat{p}_i\right)^{1 - y_i}$$

This notation looks intimidating but it just encodes the two cases above. When $y_i = 1$: the exponent on the second term is $1-1=0$, so it disappears and we are left with $\hat{p}_i^1 = \hat{p}_i$. When $y_i = 0$: the exponent on the first term is $0$, so it disappears and we are left with $(1-\hat{p}_i)^1 = 1-\hat{p}_i$. The product symbol $\prod$ means "multiply together across all observations". MLE chooses the parameters $\boldsymbol{\beta}$ that **maximise** this product: the coefficients under which the model's predicted probabilities agree most closely with the true labels across the entire training set.

**From likelihood to log-likelihood.** Multiplying many small probabilities together creates a practical problem: probabilities are numbers between 0 and 1, so multiplying hundreds of them together produces a number so tiny that a computer cannot represent it accurately, a problem called **numerical underflow**. We avoid this by taking the logarithm of the likelihood. Because $\log(a \times b) = \log(a) + \log(b)$, the product becomes a sum:

$$\ell(\boldsymbol{\beta}) = \sum_{i=1}^{n} \left[ y_i \log \hat{p}_i + (1 - y_i) \log (1 - \hat{p}_i) \right]$$

The $\sum$ symbol means "add up across all observations", which is much more numerically stable than multiplying. Crucially, because the logarithm is a monotonically increasing function (larger input always gives larger output), maximising the log-likelihood gives exactly the same answer as maximising the likelihood itself; we have not changed the problem, only the arithmetic.

**From maximisation to minimisation.** Gradient descent is set up to *minimise* a loss function: it always moves downhill. But we want to *maximise* the log-likelihood. The simplest fix is to negate it: minimising the negative log-likelihood is identical to maximising the log-likelihood. We define the **training loss** as:

$$\text{Loss} = -\ell(\boldsymbol{\beta}) = -\sum_{i=1}^{n} \left[ y_i \log \hat{p}_i + (1 - y_i) \log (1 - \hat{p}_i) \right]$$

This is also known as **binary cross-entropy**, a name you will encounter frequently in machine learning. To build intuition for what it is doing: when the model is very confident and correct (high $\hat{p}$ for a malignant tumour), $\log \hat{p}$ is close to zero and contributes almost nothing to the loss. When the model is confident and *wrong* (high $\hat{p}$ for a benign tumour), $\log(1 - \hat{p})$ becomes a large negative number, and negating it produces a large positive loss. The loss is therefore small when the model is right and very large when the model is confidently wrong. Gradient descent minimises this loss to find the optimal $\boldsymbol{\beta}$.

The cell below visualises the MLE loss landscape for the tumour classification model.

In [ ]:
# Listing 22.
%matplotlib widget
from visualisations.Figure_12 import show
show(log_reg, X_tr4, y_tr4) # Parameters created earlier

---

To make the MLE objective concrete, the cell below computes and prints the per-sample likelihood for five individual test observations. Working through these five examples by hand is the best way to build genuine intuition for what MLE is actually doing during training.

For each observation the cell shows four quantities:

- **Tumour size**: the input feature $x$, the tumour diameter in mm
- **True label**: whether the tumour is actually benign (0) or malignant (1), the ground truth the model is trying to predict
- **P(Y=1|x)**: the model's predicted probability that this specific tumour is malignant, computed by passing $z = \beta_0 + \beta_1 x$ through the sigmoid
- **Likelihood**: how probable the true label was, given the model's prediction:
    - If the tumour is truly malignant ($y=1$): likelihood $= \hat{p}$, where a high value means the model correctly assigned a high malignancy probability
    - If the tumour is truly benign ($y=0$): likelihood $= 1 - \hat{p}$, where a high value means the model correctly assigned a low malignancy probability
    - In both cases, a likelihood close to 1 means a good prediction and a likelihood close to 0 means a bad one

The **log-likelihood** column (log L) is the natural logarithm of the likelihood. Because likelihoods are probabilities between 0 and 1, their logarithms are always zero or negative: a value close to 0 means a confident correct prediction, and a large negative value means a confident wrong prediction.

At the bottom, the cell computes the **joint likelihood** (the product of all five individual likelihoods) and the **total log-likelihood** (the sum of the five log-likelihoods). The joint likelihood is the quantity that MLE maximises across the entire training set: we want the model's predicted probabilities to make the observed training labels collectively as probable as possible. Because multiplying many small probabilities together produces a number too small for a computer to represent reliably, we work with the log-likelihood (a sum) rather than the likelihood (a product), a numerical convenience that gives exactly the same answer.

The connection back to training: during fitting, the optimiser adjusts $\beta_0$ and $\beta_1$ repeatedly, recomputing these likelihoods across all 160 training observations after each update, until the total log-likelihood stops improving. The values printed below are for the test set, which the model never saw during training, but a well-fitted model should still assign high likelihoods to the correct labels on unseen data.


In [ ]:
# Listing 23.
# ── Inspect per-sample likelihoods for five illustrative test points ──────────
# Rather than looking at the total loss across all 40 test observations,
# we zoom in on five individual examples to make the MLE intuition concrete.
# For each example we can see exactly how the model's predicted probability
# contributes to the overall likelihood.
sample_idx = [0, 5, 10, 15, 20]   # indices of the five test observations to examine

# Extract the five tumour sizes and their true labels from the test set
x_samp = X_te4[sample_idx].ravel()   # tumour sizes (mm)
y_samp = y_te4[sample_idx]           # true labels (0 = benign, 1 = malignant)

# Step 1: compute z = β₀ + β₁x for each of the five observations
z_samp = log_reg.intercept_[0] + log_reg.coef_[0][0] * x_samp

# Step 2: pass z through the sigmoid to get the predicted probability of malignancy
p_samp = sigmoid(z_samp)

# Step 3: compute the likelihood contribution of each observation.
# The likelihood tells us: how probable was the true label, given the model's prediction?
#   If the true label is 1 (malignant): likelihood = p̂ᵢ
#     — we want this to be high, meaning the model correctly assigned high probability
#   If the true label is 0 (benign):   likelihood = 1 − p̂ᵢ
#     — we want this to be high, meaning the model correctly assigned low malignancy probability
# np.where(condition, value_if_true, value_if_false) selects element-by-element
likelihoods = np.where(y_samp == 1, p_samp, 1 - p_samp)

# Step 4: take the log of each likelihood.
# log-likelihood values are always ≤ 0 (since likelihoods are in (0,1] and log(1)=0).
# Values close to 0 mean the model was confident and correct;
# large negative values mean the model was confident and wrong.
log_likes = np.log(likelihoods)

# ── Print a per-sample breakdown ──────────────────────────────────────────────
print('MLE intuition: per-sample likelihoods')
print('=' * 68)
print(f'  {"Size (mm)":<12} {"True label":<16} {"P(Y=1|x)":<12} {"Likelihood":<14} {"log L"}')
print('-' * 68)
for xi, yi, pi, li, lli in zip(x_samp, y_samp, p_samp, likelihoods, log_likes):
    label_str = 'Malignant (1)' if yi == 1 else 'Benign    (0)'
    # A likelihood above 0.5 means the model assigned more than 50% probability
    # to the correct class — a reasonable prediction.
    quality = 'GOOD' if li > 0.5 else 'poor'
    print(f'  {xi:<12.1f} {label_str:<16} {pi:<12.3f} {li:<14.3f} {lli:.3f}  {quality}')

print('=' * 68)

# ── Aggregate across all five observations ────────────────────────────────────
# The joint likelihood is the product of all individual likelihoods —
# this is the quantity MLE maximises over all training observations.
# Because we are multiplying probabilities (values < 1) together, the product
# shrinks rapidly — which is why we work with log-likelihoods in practice.
import math
print(f'\nJoint likelihood (product of all five):  {math.prod(likelihoods):.6f}')

# The log-likelihood is the sum of individual log-likelihoods.
# Summing is numerically stable; multiplying tiny probabilities is not.
print(f'Log-likelihood  (sum of log Ls):         {sum(log_likes):.4f}')

print()
print('MLE goal: find β₀, β₁ that MAXIMISE the log-likelihood')
print('across ALL training observations simultaneously.')
print('Equivalently: MINIMISE the negative log-likelihood (the Loss),')
print('which is what gradient descent does during training.')

---

**Summary comparison: linear vs logistic regression**

| Property | Linear Regression | Logistic Regression |
|----------|-------------------|---------------------|
| Output | Any real number $\hat{y} \in (-\infty, +\infty)$ | Probability $\hat{p} \in (0, 1)$ |
| Decision rule | N/A (regression task) | $\hat{p} > 0.5$ → class 1 |
| Loss function | Sum of squared residuals | Negative log-likelihood (binary cross-entropy) |
| Fitting method | OLS, closed form or gradient descent | Gradient descent (MLE) |
| Space where line is fitted | Original feature space | Log-odds space |
| Key assumptions | Linearity, independence, homoscedasticity, normality | Independence; linear relationship in log-odds space |




---


## 7. Summary

🔙 [Back to Table of Contents](#Table-of-Contents)

The tables below summarise the key concepts, formulae, and sections covered in this notebook.

### Regression

| Concept | Formula | Section |
|---------|---------|---------|
| Simple linear regression | $y = \beta_0 + \beta_1 x$ | 4.1 |
| Residual | $r_i = y_i - \hat{y}_i$ | 4.1 |
| OLS training loss | $\text{Loss} = \sum_{i=1}^{n}(y_i - \hat{y}_i)^2$ | 4.2 |
| Closed-form OLS | $\beta_1 = \text{Cov}(x,y)/\text{Var}(x)$, $\quad \beta_0 = \bar{y} - \beta_1\bar{x}$ | 4.2 |
| MSE | $\frac{1}{n}\sum(y_i - \hat{y}_i)^2$ | 4.3 |
| RMSE | $\sqrt{\text{MSE}}$ | 4.3 |
| MAE | $\frac{1}{n}\sum|y_i - \hat{y}_i|$ | 4.3 |
| R² | $1 - \frac{\sum(y_i-\hat{y}_i)^2}{\sum(y_i-\bar{y})^2}$ | 4.3 |
| OLS assumptions | Linearity, independence, homoscedasticity, normality | 4.4 |
| Multiple linear regression | $y = \beta_0 + \beta_1 x_1 + \cdots + \beta_p x_p$ | 5 |

### Classification

| Concept | Formula | Section |
|---------|---------|---------|
| Sigmoid | $\sigma(z) = 1/(1+e^{-z})$ | 6.2 |
| Logit (log-odds) | $z = \log(p/(1-p))$ | 6.3 |
| Log-odds line | $z = \beta_0 + \beta_1 x_1 + \cdots + \beta_p x_p$ | 6.3–6.4 |
| Classification rule | Predict class 1 if $\sigma(z) > 0.5$ | 6.4 |
| MLE likelihood | $L(\boldsymbol{\beta}) = \prod \hat{p}_i^{\,y_i}(1-\hat{p}_i)^{1-y_i}$ | 6.6 |
| Log-likelihood | $\ell = \sum[y_i\log\hat{p}_i + (1-y_i)\log(1-\hat{p}_i)]$ | 6.6 |
| NLL loss (binary cross-entropy) | $\text{Loss} = -\ell(\boldsymbol{\beta})$ | 6.6 |


---


## 8. References

🔙 [Back to Table of Contents](#Table-of-Contents)

Bishop, C. M. (2006). *Pattern Recognition and Machine Learning*. Springer.

Hastie, T., Tibshirani, R., & Friedman, J. (2009). *The Elements of Statistical Learning* (2nd ed.). Springer. Available at: https://hastie.su.domains/ElemStatLearn/

James, G., Witten, D., Hastie, T., & Tibshirani, R. (2021). *An Introduction to Statistical Learning* (2nd ed.). Springer. Available at: https://www.statlearning.com

Pedregosa, F., et al. (2011). Scikit-learn: Machine learning in Python. *Journal of Machine Learning Research*, 12, 2825–2830.

scikit-learn developers. (n.d.). *Linear models*. scikit-learn documentation. https://scikit-learn.org/stable/modules/linear_model.html
